# EMBER end to end: §3.1 Sparse MF → §3.2 concept features → §3.3 embedding erasure

Paper: *Don't Forget Your Embeddings: Robust Knowledge Erasure via Precise Editing of Embeddings*
(Suslik, Shafran, Geva, EMNLP Findings 2026). This notebook runs the three stages of §3 for one concept,
using the paper's default hyperparameters. Each stage is split into small blocks, and each block starts
with a note on what it does.

| Step | Paper section | What happens | Repo code reused |
|---|---|---|---|
| **3.1** | Finding Embedding Features with Sparse MF | factorize $E_{V^*}^\top \approx ZY$ with WTA-sparse $Y$ | `SparseMatrixFactorization`, `fit_with_ridge` |
| **3.2** | Identifying Concept-Related Features | mass ratio $\rho_i \ge \tau$, then a two-stage Gemini judge → $F_C$ | `compute_embedding_stats`, prompts from `ember/interpret_features.py` |
| **3.3** | Erasing Concept-Related Features | $\mathbf e_t \leftarrow \mathbf e_t - \delta \sum_{i\in F_C} Y_{i,t}\,\mathbf z_i$ for concept tokens | same math as `ember/erasure/embed_edit.py` |

**Notation.** Paper $Z \in \mathbb R^{d\times k}$ = code `F_`; paper $Y \in \mathbb R^{k\times|V^*|}$ = code `G_.T`.
In this notebook, `Z` and `Y` use the paper's shapes.

**Requirements:** a GPU, `HF_TOKEN` (only needed for gated models such as Llama), and a Gemini API key for Step 3.2.

---
## 0. Setup

### 0.1 Hyperparameters (paper defaults)

| Symbol | Value | Source |
|---|---|---|
| $k$ (rank) | 100 for Gemma-2-2B-it, 200 for Llama-3.1-8B-Instruct | App. A.1 |
| $s$ (WTA sparsity on $Y$) | 0.01, so each factor keeps $\lceil 0.01\,|V^*| \rceil$ tokens | App. A.1 |
| $\lambda$ (ridge) | $10^{-4}$ | App. A.1 |
| $T$ / patience / tol | 20,000 iters / 500 iters / $10^{-4}$ | App. A.1 |
| $\|S_C\| = \|S_N\|$ | 300 sentences each (`data/*_sentences.json`) | §4.1 |
| $\tau$ (mass-ratio threshold) | 2.0 | App. A.3 |
| judge confidence | ≥ 0.85, with 20 tokens per feature | App. A.3 |
| $\delta$ (edit strength) | tuned per concept over {0.5, 1, 2, 5, 10, 50, 100, 200} | App. C.3 |

In [7]:
import gc
import json
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()  # HF_TOKEN / GEMINI_API_KEY from .env, if present
pd.set_option("display.max_colwidth", 200)

# ---- what to erase ------------------------------------------------------
CONCEPT = "Ancient Rome"
MODEL_NAME = "google/gemma-2-2b-it"     # or "meta-llama/Llama-3.1-8B-Instruct"
SEED = 42

# ---- Step 3.1: Sparse MF (App. A.1) -------------------------------------
RANK = 200 if "llama" in MODEL_NAME.lower() else 100   # k
SPARSITY = 0.01          # s
RIDGE = 1e-4             # lambda
MAX_ITER = 20_000        # T
PATIENCE = 200

# ---- Step 3.2: feature selection (App. A.3) -----------------------------
TAU = 2.0
CONFIDENCE = 0.85
TOP_K_TOKENS = 20
GEMINI_MODEL = "models/gemini-3.5-flash-lite"   # repo default; the paper used Gemini-2.5-Flash-Lite

# ---- Step 3.3: erasure (App. C.3) ---------------------------------------
DELTA_GRID = [0.5, 1.0, 2.0, 5.0, 10.0, 50.0, 100.0, 200.0]
DELTA = 10.0             # Harry Potter / Gemma value, not tuned for Ancient Rome; see the sweep in 3.3.f

concepts = [e["concept"] for e in json.loads(Path("data/concept_sentences.json").read_text())]
assert CONCEPT in concepts, f"{CONCEPT!r} is not one of: {concepts}"
print(f"concept={CONCEPT!r} | model={MODEL_NAME} | k={RANK}")

concept='Ancient Rome' | model=google/gemma-2-2b-it | k=100


### 0.2 Gemini API key

Step 3.2 is the only step that calls Gemini. You can paste your key below or put `GEMINI_API_KEY=...`
in `.env`. Clear the key from this cell before you commit or share the notebook.

In [8]:
GEMINI_API_KEY = ""  # paste here, or leave empty to use .env / the environment

if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
assert any(os.getenv(k) for k in ("GOOGLE_API_KEY", "GEMINI_API_TOKEN", "GEMINI_API_KEY")), \
    "No Gemini key found: Step 3.2 needs one."
print("Gemini key found.")

Gemini key found.


### 0.3 Repo imports

These are the building blocks the paper's pipeline uses. Reusing them means the notebook gives the same
results as `ember.train_mf_features`, `ember.interpret_features`, and `ember.run_erasure`.

In [9]:
from llm_utils.activation_generator import ActivationGenerator, extract_token_ids_sample_ids_and_labels
from transformers import AutoTokenizer

from ember.erasure.model_loader import load_hf_model
from ember.evals.mc import evaluate_mc_generation, prepare_mc_items
from ember.evals.model_wrap import WrappedHFModel
from ember.interpret_features import (
    MEMBERSHIP_PROMPT,
    TOKENS_DESC_SYS,
    SimpleGeminiClient,
    classify_membership_with_desc,
    get_description,
)
from ember.local_datasets import ConceptDataset, load_mc_qa_items, load_open_qa_examples
from ember.timing import Timer
from ember.utils import (
    SparseMatrixFactorization,
    _safe_model_name,
    build_token_label_codes,
    collect_feature_rows_for_embeddings,
    compute_embedding_stats,
    fit_with_ridge,
    get_embedding_matrix,
    get_special_token_ids,
    resolve_device,
    set_seed,
)

set_seed(SEED)
# DEVICE = resolve_device("auto")
# This host's vGPU reports CUDA as available but prohibits compute, so force CPU.
DEVICE = "cpu"
os.environ["EMBER_DEVICE"] = "cpu"  # load_hf_model reads this
print(f"device: {DEVICE}")

device: cpu


---
# Step 3.1: Finding embedding features with Sparse Matrix Factorization

**Goal:** split each token embedding $\mathbf e_t$ into a sum of a few shared directions, so that
semantically related tokens (e.g. *Harry*, *Hogwarts*, *wand*) share a feature:

$$E_{V^*}^\top \approx Z\,Y,\qquad Z\in\mathbb R^{d\times k}\ \text{(dense features)},\quad Y\in\mathbb R^{k\times|V^*|}\ \text{(WTA-sparse coefficients)}$$

The hard winner-takes-all (WTA) constraint on the rows of $Y$ means each feature $\mathbf z_i$ is used by only
1% of the tokens in $V^*$. That is why a feature can be specific to one concept.

### 3.1.a Load the model (TransformerLens)

The pipeline reads $E$ through TransformerLens. **Note:** for Gemma, TL folds the $\sqrt{d}$ embedding
multiplier into `W_E`, so the factorization runs in this *scaled* space. Step 3.3 accounts for that
when it writes back into the HF weights.

In [10]:
act_generator = ActivationGenerator(MODEL_NAME, model_device=DEVICE, data_device="cpu", mode="mlp")
tl_model = act_generator.model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

E = get_embedding_matrix(tl_model)  # (|V|, d)
VOCAB_SIZE, D_MODEL = E.shape
print(f"E: {tuple(E.shape)}  dtype={E.dtype}")

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.93it/s]


Loaded pretrained model google/gemma-2-2b-it into HookedTransformer
E: (256000, 2304)  dtype=torch.float32


### 3.1.b Build $S_C \cup S_N$ and the token set $V^*$

- $S_C$: 300 Wikipedia sentences about the concept. $S_N$: 300 neutral Wikipedia sentences.
- $V^*$: the unique non-special token ids in $S_C \cup S_N$. The model factorizes only these rows of $E$,
  which is cheaper and keeps the factorization focused on the concept. The neutral tokens let Step 3.2
  tell concept features apart from general ones.

In [11]:
# S_C / S_N resampled from Wikipedia by experiments/resample_sentences/fetch_en_sentences.py (seed 43):
# 300 sentences from the live "Ancient Rome" article, 300 neutral sentences, one per random wikimedia/wikipedia page.
RESAMPLED = Path("experiments/resample_sentences/data")
dataset = ConceptDataset(concept_name=CONCEPT, neutral_sample_seed=42,
                         concept_path=RESAMPLED / "concept_sentences_Ancient_Rome.json",
                         neutral_path=RESAMPLED / "neutral_sentences.json")
token_ids_all, sample_ids, labels_all = extract_token_ids_sample_ids_and_labels(dataset, act_generator)

special = get_special_token_ids(tl_model)
vstar_ids = sorted({int(t) for t in token_ids_all.tolist()} - special)

n_concept_sents = sum(lbl == CONCEPT for _, lbl in dataset)
print(f"|S_C| = {n_concept_sents}   |S_N| = {len(dataset) - n_concept_sents}")
print(f"token occurrences = {len(token_ids_all)}  ->  |V*| = {len(vstar_ids)} "
      f"({len(vstar_ids) / VOCAB_SIZE:.2%} of the vocabulary)")

Extracting token IDs: 100%|██████████| 120/120 [00:00<00:00, 1149.24it/s]

|S_C| = 300   |S_N| = 300
token occurrences = 15955  ->  |V*| = 4693 (1.83% of the vocabulary)


### 3.1.c Build the data matrix $A = E_{V^*}^\top$

Each column of $A$ is one token's embedding.

In [12]:
E_vstar = E.index_select(0, torch.tensor(vstar_ids, device=E.device)).detach().float()  # (|V*|, d)
A = E_vstar.T.contiguous().to(DEVICE)                                                      # (d, |V*|)

print(f"A: {tuple(A.shape)}  (d x |V*|)")
print(f"WTA keeps ceil(s*|V*|) = {math.ceil(SPARSITY * len(vstar_ids))} tokens per feature")

A: (2304, 4693)  (d x |V*|)
WTA keeps ceil(s*|V*|) = 47 tokens per feature


### 3.1.d Fit Sparse MF (alternating least squares + WTA)

The fit alternates two closed-form ridge updates:

$$Z \leftarrow A Y^\top (Y Y^\top + \lambda I)^{-1},\qquad Y \leftarrow (Z^\top Z + \lambda I)^{-1} Z^\top A$$

After each $Y$ update it applies WTA, which keeps the top-$s$ fraction of $|Y_{i,:}|$ and zeroes the rest.
It also rescales between the two factors. Initialization is $\mathcal N(0,1)$, and the fit stops early after
500 iterations without an improvement of more than $10^{-4}$. If the matrix inverse is singular,
`fit_with_ridge` retries with $\lambda \times 10$.

This is the slowest cell in Step 3.1. It prints progress every 50 iterations.

In [13]:
set_seed(SEED)
smf = SparseMatrixFactorization(RANK, fitting_device=DEVICE, g_sparsity=SPARSITY)
with Timer() as t_fit:
    ridge_used = fit_with_ridge(smf, A, MAX_ITER, patience=PATIENCE, base_reg=RIDGE)

Z = smf.F_.detach().float().cpu()      # (d, k)      embedding features z_i (columns)
Y = smf.G_.detach().float().cpu().T    # (k, |V*|)   sparse coefficients
print(f"\nfitted in {t_fit['elapsed']:.1f}s (lambda={ridge_used:g})  Z: {tuple(Z.shape)}  Y: {tuple(Y.shape)}")

Iter    0: loss=31024954.000000  (best=31024954.000000, no_improve=0)
Iter   50: loss=26630902.000000  (best=26630902.000000, no_improve=0)
Iter   66: loss=26600914.000000  (best=26600838.000000, no_improve=1)
Iter   73: loss=26592584.000000  (best=26591910.000000, no_improve=1)
Iter   84: loss=26587926.000000  (best=26586954.000000, no_improve=1)
Iter  100: loss=26587378.000000  (best=26586954.000000, no_improve=17)
Iter  150: loss=26587368.000000  (best=26586954.000000, no_improve=67)
Iter  200: loss=26587362.000000  (best=26586954.000000, no_improve=117)
Iter  250: loss=26587362.000000  (best=26586954.000000, no_improve=167)
Stopping early at iter 283 (no improvement in 200 iters)

fitted in 4.6s (lambda=0.0001)  Z: (2304, 100)  Y: (100, 4693)


### 3.1.e Check the factorization

Reconstruction quality and sparsity. Each feature should use ≈ $\lceil s|V^*|\rceil$ tokens. Tokens that
no feature covers are left as pure reconstruction error $\boldsymbol\varepsilon_t$.

In [14]:
A_cpu = A.cpu()
rel_err = (torch.norm(A_cpu - Z @ Y) / torch.norm(A_cpu)).item()
nz = Y.abs() > 0
tokens_per_feature = nz.sum(1)
features_per_token = nz.sum(0)

print(f"relative reconstruction error ||A - ZY||_F / ||A||_F = {rel_err:.4f}")
print(f"tokens per feature : min={tokens_per_feature.min().item()}  max={tokens_per_feature.max().item()}")
print(f"features per token : mean={features_per_token.float().mean():.2f}  max={features_per_token.max().item()}  "
      f"uncovered tokens={int((features_per_token == 0).sum())}")

relative reconstruction error ||A - ZY||_F / ||A||_F = 0.9143
tokens per feature : min=47  max=47
features per token : mean=1.00  max=5  uncovered tokens=1397


In [15]:
print(Z.size())
print(Y.size())
print(RANK)

torch.Size([2304, 100])
torch.Size([100, 4693])
100


### 3.1.f Baseline: truncated SVD with the same $k$

The rank-$k$ truncated SVD $A_k = U_k \Sigma_k V_k^\top$ is the best possible rank-$k$ approximation
(Eckart–Young theorem), so no rank-$k$ factorization can have a lower error, including Sparse MF.
The gap between the two errors is what the WTA sparsity costs in reconstruction.

In [16]:
U, S, Vh = torch.linalg.svd(A_cpu, full_matrices=False)
A_svd = U[:, :RANK] @ torch.diag(S[:RANK]) @ Vh[:RANK]
svd_rel_err = (torch.norm(A_cpu - A_svd) / torch.norm(A_cpu)).item()

svd_nnz = U[:, :RANK].numel() + RANK + Vh[:RANK].numel()
smf_nnz = Z.numel() + int(nz.sum())
print(f"rank-{RANK} truncated SVD : relative error = {svd_rel_err:.4f}   (parameters: {svd_nnz:,})")
print(f"rank-{RANK} Sparse MF     : relative error = {rel_err:.4f}   (nonzero parameters: {smf_nnz:,})")

# Smallest SVD rank whose error is no worse than Sparse MF's.
tail = torch.sqrt(torch.flip(torch.cumsum(torch.flip(S ** 2, [0]), 0), [0]) / (S ** 2).sum())
matching_rank = int((tail <= rel_err).nonzero()[0]) if (tail <= rel_err).any() else len(S)
print(f"SVD reaches the Sparse MF error ({rel_err:.4f}) at rank {matching_rank}")

rank-100 truncated SVD : relative error = 0.8176   (parameters: 699,800)
rank-100 Sparse MF     : relative error = 0.9143   (nonzero parameters: 235,100)
SVD reaches the Sparse MF error (0.9143) at rank 4


---
# Step 3.2: Identifying concept-related features

Most of the $k$ features are general (punctuation, function words, and so on). The selection runs in two filters:

1. **Mass ratio** $\rho_i$: is feature $i$ used mostly by concept tokens rather than neutral tokens? Keep the feature if $\rho_i \ge \tau = 2.0$.
2. **LLM judge** (Gemini, two stages): describe the feature's tokens *without* seeing the concept name,
   then ask whether that description is the concept. Keep the feature if `is_member` is true and confidence ≥ 0.85.

The features that pass both filters form $F_C$.

### 3.2.a Label every token in $V^*$

$\ell_t \in$ {concept, neutral, both}, depending on whether $t$ appears in $S_C$ only, $S_N$ only, or both.
Codes: `1` = concept, `0` = neutral, `2` = both.

In [17]:
token_to_code = build_token_label_codes(CONCEPT, dataset, tokenizer)
codes = np.array([token_to_code.get(t, 0) for t in vstar_ids])

print(f"T_C (concept-only) = {(codes == 1).sum()}   T_N (neutral-only) = {(codes == 0).sum()}   "
      f"both = {(codes == 2).sum()}")

T_C (concept-only) = 1832   T_N (neutral-only) = 2277   both = 584


### 3.2.b Score features by mass ratio and keep $\rho_i \ge \tau$

As implemented in `compute_embedding_stats` (column `ratio_abs`):

$$\rho_i = \frac{\operatorname{mean}_{t\in T_C} |Y_{i,t}|}{\operatorname{mean}_{t\in T_N} |Y_{i,t}|}$$

Tokens labeled *both* are excluded from the ratio. The threshold sweep below is the per-concept view of
Figure 6 in the paper.

In [18]:
stats = compute_embedding_stats(Y.T, vstar_ids, token_to_code, CONCEPT).sort_values("ratio_abs", ascending=False)

for thr in [1.0, 1.25, 1.5, 1.75, 2.0, 2.25]:
    print(f"rho >= {thr:<4}: {(stats['ratio_abs'] >= thr).sum():3d} / {RANK} features")

candidates = stats[stats["ratio_abs"] >= TAU]
print(f"\n{len(candidates)} candidate features pass tau = {TAU}")
candidates[["feature", "ratio_abs", "mean_abs_concept", "mean_abs_neutral", "num_concept", "num_neutral", "num_both"]]

rho >= 1.0 :  61 / 100 features
rho >= 1.25:  55 / 100 features
rho >= 1.5 :  47 / 100 features
rho >= 1.75:  40 / 100 features
rho >= 2.0 :  33 / 100 features
rho >= 2.25:  30 / 100 features

33 candidate features pass tau = 2.0


,feature,ratio_abs,mean_abs_concept,mean_abs_neutral,num_concept,num_neutral,num_both
3,3,14.161438,0.002913,0.000206,39,5,3
89,89,8.183330,0.002757,0.000337,36,9,2
85,85,7.830359,0.001074,0.000137,13,2,32
58,58,7.771319,0.003133,0.000403,40,7,0
28,28,7.127423,0.001588,0.000223,26,3,18
87,87,5.963067,0.002360,0.000396,34,6,7
98,98,5.633471,0.001893,0.000336,26,9,12
86,86,5.622236,0.002809,0.000500,36,8,3
79,79,5.480139,0.002792,0.000509,36,8,3
40,40,4.681390,0.001782,0.000381,24,7,16


### 3.2.c Collect the tokens the judge will see

For each candidate, the code takes the tokens with nonzero $Y_{i,t}$, sorted by $|Y_{i,t}|$, and keeps the top 20.
The paper text says "concept-labeled tokens", but the released code sends the top activating tokens
whatever their label. This notebook follows the code. The `labels` column shows each token's label.

In [19]:
token_label_map = {tid: "both" if c == 2 else CONCEPT if c == 1 else "Neutral" for tid, c in token_to_code.items()}
feature_rows = collect_feature_rows_for_embeddings(
    Y.T, vstar_ids, token_label_map, tl_model, _safe_model_name(MODEL_NAME), CONCEPT, RANK,
)
feat_tokens = pd.DataFrame(feature_rows).set_index("feature")[["activating_tokens", "labels"]]

cand = candidates[["feature", "ratio_abs"]].join(feat_tokens, on="feature").reset_index(drop=True)
cand["judge_tokens"] = cand["activating_tokens"].apply(lambda toks: toks[:TOP_K_TOKENS])
cand[["feature", "ratio_abs", "judge_tokens"]]

,feature,ratio_abs,judge_tokens
0,3,14.161438,"[ emperor, emperors, Emperor, king, kings, empire, queen, Empire, Emper, imperial, emper, Queen, empires, monarchs, Imperial, monarchy, monarch, kingdoms, princess, prince]"
1,89,8.183330,"[ rich, rich, wealthy, wealthiest, wealth, impoverished, poorer, poor, lavish, impover, poverty, extravagant, luxury, ornate, sophisticated, powerful, nobility, aristocracy, elite..."
2,85,7.830359,"[ along, towards, across, alongside, around, toward, against, within, through, into, along, between, throughout, among, during, from, upon, on, in, with]"
3,58,7.771319,"[ Etr, Tacitus, Etrus, Pliny, Tiberius, Josephus, Plutarch, Hadrian, Theodo, Aene, Tiber, Meiji, Carthag, Athene, Claudius, Illy, Lucius, Sulla, Pompey, Gaul]"
4,28,7.127423,"[ often, sometimes, often, Sometimes, usually, commonly, many, Many, generally, common, numerous, sometime, several, most, some, various, variously, widely, Some, few]"
5,87,5.963067,"[ complete, total, full, entire, whole, Total, full, completely, natural, fully, permanent, normal, entirely, overall, general, exact, Natural, regular, original, permanently]"
6,98,5.633471,"[ thousands, hundreds, Hundreds, dozens, thousand, innumerable, Thousand, million, centuries, numerous, miles, decades, many, kilometres, legions, hours, Many, years, several, cent..."
7,86,5.622236,"[ churches, fleets, factories, temples, libraries, schools, shops, houses, cities, companies, municipalities, ships, theatres, households, banks, communities, villas, institutions..."
8,79,5.480139,"[ confiscated, expelled, deposed, sacked, exiled, demolished, abandoned, seized, destroyed, retired, eliminated, resigned, sidelined, replaced, terminated, defeated, humiliated, r..."
9,40,4.681390,"[ went, came, took, became, grew, did, had, fell, appeared, gave, began, occurred, lacked, was, died, showed, were, remained, withdrew, threw]"


### 3.2.d-0 Rate-limited Gemini client

The free tier allows **15 requests per minute per model**. Stage 1 and Stage 2 make one request per
candidate each, so 2 × (number of candidates) requests in total. This wrapper does three things:

- spaces requests at least `60 / RPM` seconds apart (with a 10% margin);
- on a per-minute 429, waits the `retry in Ns` delay the server reports, then retries;
- fails immediately on a *daily* quota error, because waiting would not help.

Answers are cached in `descriptions` and `verdicts`. Re-running a stage only queries the features that
are still missing, so a crash never wastes quota you already spent.

In [20]:
import re
import time

GEMINI_RPM = 15


class ThrottledGemini:
    def __init__(self, inner: SimpleGeminiClient, rpm: int = GEMINI_RPM, max_attempts: int = 6):
        inner.max_retries = 1  # retries are handled here, with the server's retry delay
        self.inner = inner
        self.interval = 60.0 / rpm * 1.1
        self.max_attempts = max_attempts
        self._last = 0.0

    def generate(self, prompt: str) -> str:
        for attempt in range(self.max_attempts):
            time.sleep(max(0.0, self._last + self.interval - time.monotonic()))
            self._last = time.monotonic()
            try:
                return self.inner.generate(prompt)
            except RuntimeError as e:
                msg = str(e)
                if "429" not in msg or "PerDay" in msg or attempt == self.max_attempts - 1:
                    raise
                m = re.search(r"retry in ([\d.]+)s", msg)
                time.sleep(float(m.group(1)) + 1 if m else 60)


USE_THROTTLE = False  # billing enabled; set True to go back to free-tier pacing

if USE_THROTTLE:
    client = ThrottledGemini(SimpleGeminiClient(model_name=GEMINI_MODEL))
else:
    client = SimpleGeminiClient(model_name=GEMINI_MODEL)

# Kept across re-runs; call .clear() on both after re-fitting Step 3.1.
descriptions = globals().get("descriptions", {})
verdicts = globals().get("verdicts", {})
print(f"{2 * len(cand)} requests needed, {len(descriptions)} descriptions and {len(verdicts)} verdicts cached; "
      f"throttle={'on' if USE_THROTTLE else 'off'}")

66 requests needed, 0 descriptions and 0 verdicts cached; throttle=off


### 3.2.d LLM judge, stage 1: describe each feature *without* the concept name

Stage 1 never sees the concept name. This avoids confirmation bias, where tokens like *wizard* could
look like "Harry Potter" just because the judge was told the concept.

In [21]:
print("---- Stage 1 prompt ----")
print(TOKENS_DESC_SYS.strip(), "\n\nTokens: {tokens}\nExplanation of feature behavior:")

todo = cand[~cand["feature"].astype(int).isin(descriptions)]
for _, row in tqdm(todo.iterrows(), total=len(todo), desc="Stage 1: describe"):
    descriptions[int(row["feature"])] = get_description(client, row["judge_tokens"])
cand["description"] = cand["feature"].map(descriptions)
cand[["feature", "ratio_abs", "description"]]

---- Stage 1 prompt ----
You get tokens that represent a single feature vector (with some noise).
Infer the single most specific, cohesive concept shared by the relevant tokens.
Return only one concise sentence. No preface, no list, no caveats.

Example:
Tokens: ['▁tomorrow','▁tonight','▁yesterday','▁today','▁demain']
Explanation of feature behavior: this vector is related to specific dates and times (e.g., today/tomorrow/yesterday). 

Tokens: {tokens}
Explanation of feature behavior:


Stage 1: describe: 100%|██████████| 33/33 [00:23<00:00,  1.42it/s]


,feature,ratio_abs,description
0,3,14.161438,"This vector represents royalty, monarchy, and imperial power structures."
1,89,8.183330,"This feature vector represents the concepts of wealth, socioeconomic status, and material abundance versus poverty."
2,85,7.830359,"This vector represents spatial and temporal prepositions indicating position, movement, or relation."
3,58,7.771319,"This vector represents figures, places, and terms from ancient Roman history and classical antiquity."
4,28,7.127423,"This feature vector represents frequency, quantity, and recurrence."
5,87,5.963067,"This vector represents the concept of completeness, entirety, or uncompromised wholeness."
6,98,5.633471,"This vector represents quantities, magnitudes, and large numbers of items or time periods."
7,86,5.622236,"This feature vector represents physical or organizational entities, typically large-scale establishments, structures, or social institutions."
8,79,5.480139,"These tokens represent the concept of being forcefully removed, overthrown, or discarded from a position, place, or state of power."
9,40,4.681390,This feature represents past tense action verbs indicating completed events or states.


In [22]:
N_SHOW = 10
LABEL_TAG = {1: "C", 0: "N", 2: "B"}  # concept / neutral / both

for _, row in cand.sort_values("ratio_abs", ascending=False).iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini : {descriptions.get(feat, '<not described yet>')}")
    print("  tokens : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

=== feature 3  rho=14.16 ===
  Gemini : This vector represents royalty, monarchy, and imperial power structures.
  tokens : '▁emperor'[C] 0.26, '▁emperors'[C] 0.24, '▁Emperor'[C] 0.24, '▁king'[C] 0.22, '▁kings'[C] 0.21, '▁empire'[C] 0.21, '▁queen'[C] 0.21, '▁Empire'[C] 0.21, '▁Emper'[C] 0.21, '▁imperial'[C] 0.20

=== feature 89  rho=8.18 ===
  Gemini : This feature vector represents the concepts of wealth, socioeconomic status, and material abundance versus poverty.
  tokens : '▁rich'[C] 0.36, 'rich'[C] 0.32, '▁wealthy'[C] 0.32, '▁wealthiest'[C] 0.27, '▁wealth'[C] 0.23, '▁impoverished'[C] 0.20, '▁poorer'[C] 0.18, '▁poor'[C] 0.18, '▁lavish'[C] 0.18, '▁impover'[C] 0.16

=== feature 85  rho=7.83 ===
  Gemini : This vector represents spatial and temporal prepositions indicating position, movement, or relation.
  tokens : '▁along'[B] 0.23, '▁towards'[B] 0.20, '▁across'[C] 0.20, '▁alongside'[N] 0.20, '▁around'[C] 0.20, '▁toward'[C] 0.20, '▁against'[B] 0.18, '▁within'[B] 0.18, '▁through'[B] 0

### 3.2.e LLM judge, stage 2: is the description *the* concept?

Stage 2 sees only the concept name and the Stage 1 description. It returns `{"is_member", "confidence"}`.
A feature joins $F_C$ only if `is_member` is true **and** confidence ≥ 0.85.

In [23]:
print("---- Stage 2 prompt ----")
print(MEMBERSHIP_PROMPT.format(concept=CONCEPT, desc="{description}").strip())

todo = {f: d for f, d in descriptions.items() if f not in verdicts}
for feat, desc in tqdm(todo.items(), desc="Stage 2: classify"):
    verdicts[feat] = classify_membership_with_desc(client, CONCEPT, desc)

cand["is_member"] = cand["feature"].map(lambda f: verdicts[int(f)][0])
cand["confidence"] = pd.to_numeric(cand["feature"].map(lambda f: verdicts[int(f)][1]), errors="coerce").fillna(0.0)
cand["selected"] = (cand["is_member"] == True) & (cand["confidence"] >= CONFIDENCE)

F_C = sorted(cand.loc[cand["selected"], "feature"].astype(int).tolist())
print(f"\nF_C = {F_C}  ({len(F_C)} concept-related features)")
cand[["feature", "ratio_abs", "is_member", "confidence", "selected", "description"]]

---- Stage 2 prompt ----
You get a concept name and a feature's description.
Decide if this feature describes the given concept. Consider if the feature is DISTINCTIVE for the concept (not broad like "sport" for "basketball") and not too noisy.
A feature should be marked true only if the concept is central and dominant in the description, not just one of several themes.
Return ONLY JSON: {"is_member": true|false, "confidence": 0..1}

Example 1:
Concept: Charity
Description: This vector represents elements related to giving and donations to charitable causes.
Answer: {"is_member": true, "confidence": 0.98}

Example 2:
Concept: Charity
Description: This vector represents elements related to money, such as gambling, investing and charity.
Answer: {"is_member": false, "confidence": 0.80}

Concept: Ancient Rome
Description: {description}
Answer:


Stage 2: classify: 100%|██████████| 33/33 [00:19<00:00,  1.67it/s]


F_C = [49, 58]  (2 concept-related features)


,feature,ratio_abs,is_member,confidence,selected,description
0,3,14.161438,False,0.90,False,"This vector represents royalty, monarchy, and imperial power structures."
1,89,8.183330,False,0.95,False,"This feature vector represents the concepts of wealth, socioeconomic status, and material abundance versus poverty."
2,85,7.830359,False,0.99,False,"This vector represents spatial and temporal prepositions indicating position, movement, or relation."
3,58,7.771319,True,0.99,True,"This vector represents figures, places, and terms from ancient Roman history and classical antiquity."
4,28,7.127423,False,1.00,False,"This feature vector represents frequency, quantity, and recurrence."
5,87,5.963067,False,0.99,False,"This vector represents the concept of completeness, entirety, or uncompromised wholeness."
6,98,5.633471,False,0.99,False,"This vector represents quantities, magnitudes, and large numbers of items or time periods."
7,86,5.622236,False,0.95,False,"This feature vector represents physical or organizational entities, typically large-scale establishments, structures, or social institutions."
8,79,5.480139,False,0.95,False,"These tokens represent the concept of being forcefully removed, overthrown, or discarded from a position, place, or state of power."
9,40,4.681390,False,0.99,False,This feature represents past tense action verbs indicating completed events or states.


In [24]:
is_true = cand["is_member"] == True
ordered = cand.assign(
    _group=(~is_true).astype(int),
    _key=np.where(is_true, -cand["confidence"], cand["confidence"]),
).sort_values(["_group", "_key"])

for _, row in ordered.iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    is_member, conf = verdicts.get(feat, (None, None))
    verdict = "<not classified yet>" if feat not in verdicts else (
        f"is_member={is_member}  confidence={conf if conf is not None else 'n/a'}  "
        f"-> {'SELECTED' if row['selected'] else 'rejected'}"
    )

    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini  : {descriptions.get(feat, '<not described yet>')}")
    print(f"  verdict : {verdict}")
    print("  tokens  : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

=== feature 58  rho=7.77 ===
  Gemini  : This vector represents figures, places, and terms from ancient Roman history and classical antiquity.
  verdict : is_member=True  confidence=0.99  -> SELECTED
  tokens  : '▁Etr'[C] 0.23, '▁Tacitus'[C] 0.23, '▁Etrus'[C] 0.21, '▁Pliny'[C] 0.21, '▁Tiberius'[C] 0.20, '▁Josephus'[C] 0.20, '▁Plutarch'[C] 0.19, '▁Hadrian'[C] 0.19, '▁Theodo'[C] 0.18, '▁Aene'[C] 0.17

=== feature 49  rho=3.26 ===
  Gemini  : This feature represents the city of Rome, its historical empire, and its associated cultural and religious context.
  verdict : is_member=True  confidence=0.95  -> SELECTED
  tokens  : '▁Roman'[C] 0.35, 'Roman'[C] 0.32, '▁Romans'[C] 0.31, '▁Rome'[C] 0.30, '▁Rom'[C] 0.29, 'Romans'[C] 0.27, 'Rome'[C] 0.27, '▁Roma'[N] 0.26, '▁Romana'[C] 0.26, 'rom'[N] 0.20

=== feature 3  rho=14.16 ===
  Gemini  : This vector represents royalty, monarchy, and imperial power structures.
  verdict : is_member=False  confidence=0.9  -> rejected
  tokens  : '▁emperor'[C] 0.

In [25]:
members = cand[cand["is_member"] == True].sort_values("ratio_abs", ascending=False)
print(f"{len(members)} / {len(cand)} candidates have is_member=True\n")

for _, row in members.iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini  : {row['description']}")
    print(f"  verdict : confidence={row['confidence']:.2f}  -> {'SELECTED' if row['selected'] else 'rejected (confidence < ' + str(CONFIDENCE) + ')'}")
    print("  tokens  : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

2 / 33 candidates have is_member=True

=== feature 58  rho=7.77 ===
  Gemini  : This vector represents figures, places, and terms from ancient Roman history and classical antiquity.
  verdict : confidence=0.99  -> SELECTED
  tokens  : '▁Etr'[C] 0.23, '▁Tacitus'[C] 0.23, '▁Etrus'[C] 0.21, '▁Pliny'[C] 0.21, '▁Tiberius'[C] 0.20, '▁Josephus'[C] 0.20, '▁Plutarch'[C] 0.19, '▁Hadrian'[C] 0.19, '▁Theodo'[C] 0.18, '▁Aene'[C] 0.17

=== feature 49  rho=3.26 ===
  Gemini  : This feature represents the city of Rome, its historical empire, and its associated cultural and religious context.
  verdict : confidence=0.95  -> SELECTED
  tokens  : '▁Roman'[C] 0.35, 'Roman'[C] 0.32, '▁Romans'[C] 0.31, '▁Rome'[C] 0.30, '▁Rom'[C] 0.29, 'Romans'[C] 0.27, 'Rome'[C] 0.27, '▁Roma'[N] 0.26, '▁Romana'[C] 0.26, 'rom'[N] 0.20



### 3.2.f Inspect the selected features

Table 3 of the paper reports only 1–3 embedding features per concept (Harry Potter on Gemma: 1).
Each selected feature should be a tight cluster of concept-specific tokens.

In [26]:
if not F_C:
    print("No feature passed both filters: Step 3.3 will be a no-op for this concept.")

for feat in F_C:
    row = cand.loc[cand["feature"] == feat].iloc[0]
    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f}  conf={row['confidence']:.2f} ===")
    print(f"  {row['description']}")
    print("  tokens:", ", ".join(f"{t!r}[{l[0]}]" for t, l in zip(row["activating_tokens"][:30], row["labels"][:30])))
    print()

=== feature 49  rho=3.26  conf=0.95 ===
  This feature represents the city of Rome, its historical empire, and its associated cultural and religious context.
  tokens: ' Roman'[A], 'Roman'[A], ' Romans'[A], ' Rome'[A], ' Rom'[A], 'Romans'[A], 'Rome'[A], ' Roma'[N], ' Romana'[A], 'rom'[N], ' Caesar'[A], ' Vatican'[N], 'Caesar'[A], ' Romania'[A], ' Romanian'[N], ' Christians'[A], ' Christian'[A], ' Nero'[A], 'losseum'[A], 'dom'[N], ' Titus'[A], ' Caesars'[A], ' RC'[N], 'Titus'[A], 'oman'[A], 'Christian'[N], ' Dom'[A], ' Church'[N], ' toga'[A], ' Rogers'[N]

=== feature 58  rho=7.77  conf=0.99 ===
  This vector represents figures, places, and terms from ancient Roman history and classical antiquity.
  tokens: ' Etr'[A], ' Tacitus'[A], ' Etrus'[A], ' Pliny'[A], ' Tiberius'[A], ' Josephus'[A], ' Plutarch'[A], ' Hadrian'[A], ' Theodo'[A], ' Aene'[A], ' Tiber'[A], ' Meiji'[N], ' Carthag'[A], ' Athene'[N], ' Claudius'[A], ' Illy'[A], ' Lucius'[A], ' Sulla'[A], ' Pompey'[A], ' Gaul'[A], ' Dalai

In [54]:
# One column per selected feature: every token with nonzero Y_{i,t}, sorted by |Y_{i,t}|.
columns = {}
for feat in F_C:
    mags = Y[feat].abs()
    nz_idx = torch.nonzero(mags > 0).flatten()
    nz_idx = nz_idx[torch.argsort(mags[nz_idx], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in nz_idx])
    columns[f"feature {feat}"] = pd.Series(
        [f"{t} [{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, nz_idx)]
    )

wide = pd.DataFrame(columns).fillna("")
wide.index = range(1, len(wide) + 1)

all_cols = sorted({j for feat in F_C for j in torch.nonzero(Y[feat].abs() > 0).flatten().tolist()})
print(f"{len(F_C)} concept features, {len(all_cols)} distinct tokens "
      f"(C={sum(codes[j] == 1 for j in all_cols)}, N={sum(codes[j] == 0 for j in all_cols)}, "
      f"B={sum(codes[j] == 2 for j in all_cols)})")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(wide)

2 concept features, 92 distinct tokens (C=67, N=24, B=1)


,feature 49,feature 58
1,▁Roman [C] 0.35,▁Etr [C] 0.23
2,Roman [C] 0.32,▁Tacitus [C] 0.23
3,▁Romans [C] 0.31,▁Etrus [C] 0.21
4,▁Rome [C] 0.30,▁Pliny [C] 0.21
5,▁Rom [C] 0.29,▁Tiberius [C] 0.20
6,Romans [C] 0.27,▁Josephus [C] 0.20
7,Rome [C] 0.27,▁Plutarch [C] 0.19
8,▁Roma [N] 0.26,▁Hadrian [C] 0.19
9,▁Romana [C] 0.26,▁Theodo [C] 0.18
10,rom [N] 0.20,▁Aene [C] 0.17


In [55]:
TOP_N = 30
E_ref = (E_vstar_tl if "E_vstar_tl" in globals() else E_vstar).float().cpu()  # E_vstar is deleted in 3.3.a
token_strs = tokenizer.convert_ids_to_tokens(vstar_ids)

group_masks = {
    "C": codes == 1,
    "C / B": (codes == 1) | (codes == 2),
    "C / B / N": np.ones_like(codes, dtype=bool),
}

for feat in F_C:
    y = Y[feat]
    # z's sign is arbitrary (Y can absorb it); orient it so its support tokens mostly have positive Y.
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    dots = E_ref @ z
    cos = dots / (E_ref.norm(dim=1) * z.norm())

    df = pd.DataFrame({
        "token": token_strs,
        "label": [LABEL_TAG[c] for c in codes],
        "dot": dots.numpy(),
        "cos": cos.numpy(),
        "Y": (sign * y).numpy(),
        "in_support": (y != 0).numpy(),
    })

    print(f"=== feature {feat}   ||z|| = {z.norm():.3f}   sign flipped: {sign < 0} ===")
    for name, mask in group_masks.items():
        top = df[mask].sort_values("dot", ascending=False).head(TOP_N).reset_index(drop=True)
        top.index += 1
        print(f"\n-- {name}: {int(mask.sum())} tokens, top {TOP_N} by <e_t, z> --")
        display(top)

=== feature 49   ||z|| = 211.620   sign flipped: True ===

-- C: 1832 tokens, top 30 by <e_t, z> --


,token,label,dot,cos,Y,in_support
1,▁Romans,C,14266.778320,0.751493,0.309078,True
2,▁Romana,C,12451.606445,0.630603,0.255492,True
3,Roman,C,12136.962891,0.749619,0.324827,True
4,▁Roman,C,11776.331055,0.702892,0.354725,True
5,Romans,C,11775.359375,0.691290,0.268974,True
6,Rome,C,11719.596680,0.704553,0.267889,True
7,▁Etrus,C,11688.729492,0.447040,0.000000,False
8,▁Etr,C,11352.501953,0.425653,-0.000000,False
9,▁Tacitus,C,11287.577148,0.436854,0.000000,False
10,▁Rome,C,11196.457031,0.682738,0.300522,True



-- C / B: 2416 tokens, top 30 by <e_t, z> --


,token,label,dot,cos,Y,in_support
1,▁Romans,C,14266.778320,0.751493,0.309078,True
2,▁Romana,C,12451.606445,0.630603,0.255492,True
3,Roman,C,12136.962891,0.749619,0.324827,True
4,▁Roman,C,11776.331055,0.702892,0.354725,True
5,Romans,C,11775.359375,0.691290,0.268974,True
6,Rome,C,11719.596680,0.704553,0.267889,True
7,▁Etrus,C,11688.729492,0.447040,0.000000,False
8,▁Etr,C,11352.501953,0.425653,-0.000000,False
9,▁Tacitus,C,11287.577148,0.436854,0.000000,False
10,▁Rome,C,11196.457031,0.682738,0.300522,True



-- C / B / N: 4693 tokens, top 30 by <e_t, z> --


...

,token,label,dot,cos,Y,in_support
1,▁Etrus,C,49189.066406,0.810277,0.211500,True
2,▁Etr,C,48983.097656,0.791037,0.231422,True
3,▁Tacitus,C,47554.898438,0.792714,0.225946,True
4,▁Pliny,C,45486.085938,0.771069,0.210809,True
5,▁Josephus,C,43754.625000,0.759654,0.196367,True
6,▁Plutarch,C,42644.261719,0.770011,0.191499,True
7,▁Theodo,C,42094.625000,0.725944,0.176367,True
8,▁Teut,C,41868.160156,0.741602,0.133078,True
9,▁Persians,C,41301.167969,0.755874,0.125038,True
10,▁Aene,C,39837.328125,0.718191,0.170950,True



-- C / B: 2416 tokens, top 30 by <e_t, z> --


,token,label,dot,cos,Y,in_support
1,▁Etrus,C,49189.066406,0.810277,0.211500,True
2,▁Etr,C,48983.097656,0.791037,0.231422,True
3,▁Tacitus,C,47554.898438,0.792714,0.225946,True
4,▁Pliny,C,45486.085938,0.771069,0.210809,True
5,▁Josephus,C,43754.625000,0.759654,0.196367,True
6,▁Plutarch,C,42644.261719,0.770011,0.191499,True
7,▁Theodo,C,42094.625000,0.725944,0.176367,True
8,▁Teut,C,41868.160156,0.741602,0.133078,True
9,▁Persians,C,41301.167969,0.755874,0.125038,True
10,▁Aene,C,39837.328125,0.718191,0.170950,True



-- C / B / N: 4693 tokens, top 30 by <e_t, z> --


,token,label,dot,cos,Y,in_support
1,▁Etrus,C,49189.066406,0.810277,0.211500,True
2,▁Etr,C,48983.097656,0.791037,0.231422,True
3,▁Tacitus,C,47554.898438,0.792714,0.225946,True
4,▁Pliny,C,45486.085938,0.771069,0.210809,True
5,▁Josephus,C,43754.625000,0.759654,0.196367,True
6,▁Plutarch,C,42644.261719,0.770011,0.191499,True
7,▁Theodo,C,42094.625000,0.725944,0.176367,True
8,▁Teut,C,41868.160156,0.741602,0.133078,True
9,▁Persians,C,41301.167969,0.755874,0.125038,True
10,▁Meiji,N,40459.781250,0.737822,0.156999,True


In [29]:
E_unit = E_ref / E_ref.norm(dim=1, keepdim=True)

for feat in F_C:
    y = Y[feat]
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    dots_unit = E_unit @ z

    df_unit = pd.DataFrame({
        "token": token_strs,
        "label": [LABEL_TAG[c] for c in codes],
        "dot_unit": dots_unit.numpy(),
        "||e_t||": E_ref.norm(dim=1).numpy(),
        "Y": (sign * y).numpy(),
        "in_support": (y != 0).numpy(),
    })

    print(f"=== feature {feat}   ||z|| = {z.norm():.3f}   sign flipped: {sign < 0} ===")
    for name, mask in group_masks.items():
        top = df_unit[mask].sort_values("dot_unit", ascending=False).head(TOP_N).reset_index(drop=True)
        top.index += 1
        print(f"\n-- {name}: {int(mask.sum())} tokens, top {TOP_N} by <e_t / ||e_t||, z> --")
        display(top)

=== feature 49   ||z|| = 211.620   sign flipped: True ===

-- C: 1832 tokens, top 30 by <e_t / ||e_t||, z> --


,token,label,dot_unit,||e_t||,Y,in_support
1,▁Romans,C,159.030640,89.710884,0.309078,True
2,Roman,C,158.634109,76.509178,0.324827,True
3,Rome,C,149.097198,78.603722,0.267889,True
4,▁Roman,C,148.745758,79.170883,0.354725,True
5,Romans,C,146.290512,80.492981,0.268974,True
6,▁Rome,C,144.480850,77.494415,0.300522,True
7,▁Romana,C,133.448135,93.306702,0.255492,True
8,▁Rom,C,120.253685,81.602921,0.286808,True
9,▁Romania,C,104.519661,79.486275,0.100338,True
10,▁Caesar,C,100.959572,83.167336,0.112425,True



-- C / B: 2416 tokens, top 30 by <e_t / ||e_t||, z> --


,token,label,dot_unit,||e_t||,Y,in_support
1,▁Romans,C,159.030640,89.710884,0.309078,True
2,Roman,C,158.634109,76.509178,0.324827,True
3,Rome,C,149.097198,78.603722,0.267889,True
4,▁Roman,C,148.745758,79.170883,0.354725,True
5,Romans,C,146.290512,80.492981,0.268974,True
6,▁Rome,C,144.480850,77.494415,0.300522,True
7,▁Romana,C,133.448135,93.306702,0.255492,True
8,▁Rom,C,120.253685,81.602921,0.286808,True
9,▁Romania,C,104.519661,79.486275,0.100338,True
10,▁Caesar,C,100.959572,83.167336,0.112425,True



-- C / B / N: 4693 tokens, top 30 by <e_t / ||e_t||, z> --


...

,token,label,dot_unit,||e_t||,Y,in_support
1,▁Etrus,C,398.110504,123.556320,0.211500,True
2,▁Tacitus,C,389.481354,122.098045,0.225946,True
3,▁Etr,C,388.657349,126.031570,0.231422,True
4,▁Pliny,C,378.846680,120.064613,0.210809,True
5,▁Plutarch,C,378.326630,112.718102,0.191499,True
6,▁Josephus,C,373.238251,117.229736,0.196367,True
7,▁Persians,C,371.380585,111.209801,0.125038,True
8,▁Teut,C,364.368317,114.906120,0.133078,True
9,▁barbarians,C,360.687103,107.871559,0.000000,False
10,▁Tiberius,C,358.926605,109.632141,0.202523,True



-- C / B: 2416 tokens, top 30 by <e_t / ||e_t||, z> --


,token,label,dot_unit,||e_t||,Y,in_support
1,▁Etrus,C,398.110504,123.556320,0.211500,True
2,▁Tacitus,C,389.481354,122.098045,0.225946,True
3,▁Etr,C,388.657349,126.031570,0.231422,True
4,▁Pliny,C,378.846680,120.064613,0.210809,True
5,▁Plutarch,C,378.326630,112.718102,0.191499,True
6,▁Josephus,C,373.238251,117.229736,0.196367,True
7,▁Persians,C,371.380585,111.209801,0.125038,True
8,▁Teut,C,364.368317,114.906120,0.133078,True
9,▁barbarians,C,360.687103,107.871559,0.000000,False
10,▁Tiberius,C,358.926605,109.632141,0.202523,True



-- C / B / N: 4693 tokens, top 30 by <e_t / ||e_t||, z> --


,token,label,dot_unit,||e_t||,Y,in_support
1,▁Etrus,C,398.110504,123.556320,0.211500,True
2,▁Tacitus,C,389.481354,122.098045,0.225946,True
3,▁Etr,C,388.657349,126.031570,0.231422,True
4,▁Pliny,C,378.846680,120.064613,0.210809,True
5,▁Plutarch,C,378.326630,112.718102,0.191499,True
6,▁Josephus,C,373.238251,117.229736,0.196367,True
7,▁Persians,C,371.380585,111.209801,0.125038,True
8,▁Teut,C,364.368317,114.906120,0.133078,True
9,▁Meiji,N,362.511261,111.609726,0.156999,True
10,▁barbarians,C,360.687103,107.871559,0.000000,False


In [30]:
TOP_K_COUNT = 100

rows = []
for feat in F_C:
    y = Y[feat]
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    rankings = {
        "<e_t, z>": E_ref @ z,
        "<e_t / ||e_t||, z>": E_unit @ z,
    }
    for name, scores in rankings.items():
        top = torch.argsort(scores, descending=True)[:TOP_K_COUNT].numpy()
        rows.append({
            "feature": feat,
            "ranking": name,
            "C": int((codes[top] == 1).sum()),
            "B": int((codes[top] == 2).sum()),
            "N": int((codes[top] == 0).sum()),
            "C in support": int(((codes[top] == 1) & (y[top] != 0).numpy()).sum()),
        })

print(f"Label counts among the top {TOP_K_COUNT} of all {len(vstar_ids)} tokens in V* "
      f"(base rate: C={(codes == 1).mean():.1%}, B={(codes == 2).mean():.1%}, N={(codes == 0).mean():.1%})")
pd.DataFrame(rows)

Label counts among the top 100 of all 4693 tokens in V* (base rate: C=39.0%, B=12.4%, N=48.5%)


,feature,ranking,C,B,N,C in support
0,49,"<e_t, z>",77,0,23,17
1,49,"<e_t / ||e_t||, z>",79,0,21,21
2,58,"<e_t, z>",72,0,28,37
3,58,"<e_t / ||e_t||, z>",70,0,30,38


In [31]:
# ---- Dense refit: fix Z, solve Y' = argmin ||E*^T - Z Y'||_F (no sparsity) ----
TOP_DIRS = 5
AR_FEATS = list(F_C)          # the Ancient Rome concept column(s) of Z; override e.g. [37] if you want one
AR_MARK = "★★★ AR ★★★"

A_ref = (E_vstar_tl if "E_vstar_tl" in globals() else E_vstar).double().cpu().T   # (d, |V*|)
Z64 = Z.double()                                                                    # (d, k)
Y_dense = torch.linalg.lstsq(Z64, A_ref).solution.float()                           # (k, |V*|)

dense_err = (torch.norm(A_ref - Z64 @ Y_dense.double()) / torch.norm(A_ref)).item()
sparse_err = (torch.norm(A_ref - Z64 @ Y.double()) / torch.norm(A_ref)).item()
print(f"relative error: sparse Y = {sparse_err:.4f}   dense Y' = {dense_err:.4f}")
print(f"AR feature(s): {AR_FEATS}   (||z|| = {[round(Z[:, f].norm().item(), 3) for f in AR_FEATS]})\n")

for ar in AR_FEATS:
    mags = Y[ar].abs()
    support = torch.nonzero(mags > 0).flatten()
    support = support[torch.argsort(mags[support], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in support])

    print(f"{'=' * 100}\n=== AR feature {ar}: {len(support)} tokens with nonzero Y, by decreasing |Y| ===\n{'=' * 100}")
    for rank_t, (j, tok) in enumerate(zip(support, toks), 1):
        y_dense_t = Y_dense[:, j]
        order = torch.argsort(y_dense_t.abs(), descending=True).tolist()
        top = order[:TOP_DIRS]

        print(f"\n{rank_t:3d}. {tok!r} [{LABEL_TAG[codes[j]]}]   sparse Y[{ar}] = {Y[ar, j]:+.3f}")
        for r, f in enumerate(top, 1):
            mark = f"   <-- {AR_MARK}" if f in AR_FEATS else ""
            print(f"       #{r}  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}{mark}")
        for f in AR_FEATS:
            if f not in top:
                print(f"       ..  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}   <-- {AR_MARK} (rank {order.index(f) + 1}/{len(order)})")

relative error: sparse Y = 0.9144   dense Y' = 0.8387
AR feature(s): [49, 58]   (||z|| = [211.62, 491.326])

=== AR feature 49: 47 tokens with nonzero Y, by decreasing |Y| ===

  1. '▁Roman' [C]   sparse Y[49] = -0.355
       #1  z_49   Y' = -0.355   <-- ★★★ AR ★★★
       #2  z_42   Y' = -0.052
       #3  z_58   Y' = -0.038   <-- ★★★ AR ★★★
       #4  z_68   Y' = +0.031
       #5  z_18   Y' = +0.024

  2. 'Roman' [C]   sparse Y[49] = -0.325
       #1  z_49   Y' = -0.325   <-- ★★★ AR ★★★
       #2  z_42   Y' = -0.053
       #3  z_68   Y' = +0.029
       #4  z_58   Y' = -0.023   <-- ★★★ AR ★★★
       #5  z_56   Y' = -0.020

  3. '▁Romans' [C]   sparse Y[49] = -0.309
       #1  z_49   Y' = -0.309   <-- ★★★ AR ★★★
       #2  z_48   Y' = +0.047
       #3  z_58   Y' = +0.042   <-- ★★★ AR ★★★
       #4  z_18   Y' = +0.038
       #5  z_19   Y' = -0.020

  4. '▁Rome' [C]   sparse Y[49] = -0.301
       #1  z_49   Y' = -0.301   <-- ★★★ AR ★★★
       #2  z_68   Y' = -0.067
       #3  z_93   Y' = +

In [32]:
# ---- Tokens where the AR direction dominates Y': top-1 is AR and |Y'_1| >= RATIO * |Y'_2| ----
RATIO = 2.0

for ar in AR_FEATS:
    mags = Y[ar].abs()
    support = torch.nonzero(mags > 0).flatten()
    support = support[torch.argsort(mags[support], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in support])

    kept = []
    for rank_t, (j, tok) in enumerate(zip(support, toks), 1):
        y_dense_t = Y_dense[:, j]
        order = torch.argsort(y_dense_t.abs(), descending=True).tolist()
        first, second = y_dense_t[order[0]].abs().item(), y_dense_t[order[1]].abs().item()
        if order[0] == ar and first >= RATIO * second:
            kept.append((rank_t, j, tok, order, first / second))

    print(f"{'=' * 100}\n=== AR feature {ar}: {len(kept)} / {len(support)} tokens where z_{ar} is top-1 "
          f"and |Y'| >= {RATIO:g}x the runner-up ===\n{'=' * 100}")
    for rank_t, j, tok, order, ratio in kept:
        y_dense_t = Y_dense[:, j]
        print(f"\n{rank_t:3d}. {tok!r} [{LABEL_TAG[codes[j]]}]   sparse Y[{ar}] = {Y[ar, j]:+.3f}   "
              f"top-1/top-2 = {ratio:.2f}x")
        for r, f in enumerate(order[:TOP_DIRS], 1):
            mark = f"   <-- {AR_MARK}" if f in AR_FEATS else ""
            print(f"       #{r}  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}{mark}")

=== AR feature 49: 13 / 47 tokens where z_49 is top-1 and |Y'| >= 2x the runner-up ===

  1. '▁Roman' [C]   sparse Y[49] = -0.355   top-1/top-2 = 6.77x
       #1  z_49   Y' = -0.355   <-- ★★★ AR ★★★
       #2  z_42   Y' = -0.052
       #3  z_58   Y' = -0.038   <-- ★★★ AR ★★★
       #4  z_68   Y' = +0.031
       #5  z_18   Y' = +0.024

  2. 'Roman' [C]   sparse Y[49] = -0.325   top-1/top-2 = 6.14x
       #1  z_49   Y' = -0.325   <-- ★★★ AR ★★★
       #2  z_42   Y' = -0.053
       #3  z_68   Y' = +0.029
       #4  z_58   Y' = -0.023   <-- ★★★ AR ★★★
       #5  z_56   Y' = -0.020

  3. '▁Romans' [C]   sparse Y[49] = -0.309   top-1/top-2 = 6.63x
       #1  z_49   Y' = -0.309   <-- ★★★ AR ★★★
       #2  z_48   Y' = +0.047
       #3  z_58   Y' = +0.042   <-- ★★★ AR ★★★
       #4  z_18   Y' = +0.038
       #5  z_19   Y' = -0.020

  4. '▁Rome' [C]   sparse Y[49] = -0.301   top-1/top-2 = 4.51x
       #1  z_49   Y' = -0.301   <-- ★★★ AR ★★★
       #2  z_68   Y' = -0.067
       #3  z_93   Y' = +0

In [33]:
# ---- Tokens skipped by the previous cell: AR is not top-1, or top-1 but < RATIO x the runner-up ----
for ar in AR_FEATS:
    mags = Y[ar].abs()
    support = torch.nonzero(mags > 0).flatten()
    support = support[torch.argsort(mags[support], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in support])

    skipped = []
    for rank_t, (j, tok) in enumerate(zip(support, toks), 1):
        y_dense_t = Y_dense[:, j]
        order = torch.argsort(y_dense_t.abs(), descending=True).tolist()
        first, second = y_dense_t[order[0]].abs().item(), y_dense_t[order[1]].abs().item()
        if order[0] != ar:
            skipped.append((rank_t, j, tok, order, f"AR is not top-1 (rank {order.index(ar) + 1}/{len(order)})"))
        elif first < RATIO * second:
            skipped.append((rank_t, j, tok, order, f"AR is top-1 but only {first / second:.2f}x the runner-up"))

    n_not_top = sum(r.startswith("AR is not") for *_, r in skipped)
    print(f"{'=' * 100}\n=== AR feature {ar}: {len(skipped)} / {len(support)} tokens skipped "
          f"({n_not_top} AR not top-1, {len(skipped) - n_not_top} top-1 but < {RATIO:g}x) ===\n{'=' * 100}")
    for rank_t, j, tok, order, reason in skipped:
        y_dense_t = Y_dense[:, j]
        print(f"\n{rank_t:3d}. {tok!r} [{LABEL_TAG[codes[j]]}]   sparse Y[{ar}] = {Y[ar, j]:+.3f}   -> {reason}")
        for r, f in enumerate(order[:TOP_DIRS], 1):
            mark = f"   <-- {AR_MARK}" if f in AR_FEATS else ""
            print(f"       #{r}  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}{mark}")
        if ar not in order[:TOP_DIRS]:
            print(f"       ..  z_{ar:<4d} Y' = {y_dense_t[ar]:+.3f}   <-- {AR_MARK} (rank {order.index(ar) + 1}/{len(order)})")

=== AR feature 49: 34 / 47 tokens skipped (17 AR not top-1, 17 top-1 but < 2x) ===

 13. 'Caesar' [C]   sparse Y[49] = -0.101   -> AR is top-1 but only 1.68x the runner-up
       #1  z_49   Y' = -0.101   <-- ★★★ AR ★★★
       #2  z_58   Y' = +0.060   <-- ★★★ AR ★★★
       #3  z_18   Y' = -0.030
       #4  z_59   Y' = -0.028
       #5  z_3    Y' = -0.027

 14. '▁Romania' [C]   sparse Y[49] = -0.100   -> AR is not top-1 (rank 2/100)
       #1  z_68   Y' = -0.155
       #2  z_49   Y' = -0.100   <-- ★★★ AR ★★★
       #3  z_19   Y' = -0.039
       #4  z_18   Y' = -0.024
       #5  z_93   Y' = +0.021

 15. '▁Romanian' [N]   sparse Y[49] = -0.091   -> AR is not top-1 (rank 2/100)
       #1  z_42   Y' = -0.198
       #2  z_49   Y' = -0.091   <-- ★★★ AR ★★★
       #3  z_19   Y' = -0.039
       #4  z_93   Y' = +0.025
       #5  z_92   Y' = +0.021

 16. '▁Christians' [C]   sparse Y[49] = -0.079   -> AR is not top-1 (rank 2/100)
       #1  z_48   Y' = +0.128
       #2  z_49   Y' = -0.079   <-- ★★★

In [34]:
# ---- All C-labelled tokens, sorted by |Y'_AR| / max_{i != AR} |Y'_i| (= top-1/top-2 when AR is top-1) ----
for ar in AR_FEATS:
    c_cols = np.where(codes == 1)[0].tolist()
    rows = []
    for j in c_cols:
        y_dense_t = Y_dense[:, j]
        others_max = torch.cat([y_dense_t[:ar], y_dense_t[ar + 1:]]).abs().max().item()
        rows.append((y_dense_t[ar].abs().item() / others_max, j))
    rows.sort(reverse=True)

    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for _, j in rows])
    n_pass = sum(r >= RATIO for r, _ in rows)
    n_top1 = sum(r >= 1.0 for r, _ in rows)
    print(f"{'=' * 100}\n=== AR feature {ar}: {len(rows)} C tokens | AR top-1: {n_top1} | "
          f"AR top-1 and >= {RATIO:g}x: {n_pass} ===\n{'=' * 100}")

    shown_ratio_line = shown_top1_line = False
    for rank_t, ((ratio, j), tok) in enumerate(zip(rows, toks), 1):
        if not shown_ratio_line and ratio < RATIO:
            print(f"\n{'-' * 40} below {RATIO:g}x {'-' * 40}")
            shown_ratio_line = True
        if not shown_top1_line and ratio < 1.0:
            print(f"\n{'-' * 36} AR no longer top-1 {'-' * 36}")
            shown_top1_line = True

        y_dense_t = Y_dense[:, j]
        order = torch.argsort(y_dense_t.abs(), descending=True).tolist()
        in_support = "in AR support" if Y[ar, j] != 0 else "not in AR support"
        print(f"\n{rank_t:3d}. {tok!r} [C]   ratio = {ratio:.2f}x   sparse Y[{ar}] = {Y[ar, j]:+.3f}  ({in_support})")
        for r, f in enumerate(order[:TOP_DIRS], 1):
            mark = f"   <-- {AR_MARK}" if f in AR_FEATS else ""
            print(f"       #{r}  z_{f:<4d} Y' = {y_dense_t[f]:+.3f}{mark}")
        if ar not in order[:TOP_DIRS]:
            print(f"       ..  z_{ar:<4d} Y' = {y_dense_t[ar]:+.3f}   <-- {AR_MARK} (rank {order.index(ar) + 1}/{len(order)})")

=== AR feature 49: 1832 C tokens | AR top-1: 19 | AR top-1 and >= 2x: 10 ===

  1. '▁Romana' [C]   ratio = 11.84x   sparse Y[49] = -0.255  (in AR support)
       #1  z_49   Y' = -0.255   <-- ★★★ AR ★★★
       #2  z_29   Y' = +0.022
       #3  z_18   Y' = -0.020
       #4  z_61   Y' = -0.019
       #5  z_70   Y' = -0.019

  2. 'Romans' [C]   ratio = 9.67x   sparse Y[49] = -0.269  (in AR support)
       #1  z_49   Y' = -0.269   <-- ★★★ AR ★★★
       #2  z_18   Y' = -0.028
       #3  z_48   Y' = +0.027
       #4  z_90   Y' = -0.019
       #5  z_24   Y' = +0.019

  3. '▁Roman' [C]   ratio = 6.77x   sparse Y[49] = -0.355  (in AR support)
       #1  z_49   Y' = -0.355   <-- ★★★ AR ★★★
       #2  z_42   Y' = -0.052
       #3  z_58   Y' = -0.038   <-- ★★★ AR ★★★
       #4  z_68   Y' = +0.031
       #5  z_18   Y' = +0.024

  4. '▁Romans' [C]   ratio = 6.63x   sparse Y[49] = -0.309  (in AR support)
       #1  z_49   Y' = -0.309   <-- ★★★ AR ★★★
       #2  z_48   Y' = +0.047
       #3  z_58   Y' 

In [35]:
# ---- How orthogonal is each AR direction to the span of the other Z columns? ----
Z64 = Z.double()
k = Z64.shape[1]


def angle_to_span(f: int, exclude=()) -> dict:
    """Angle between z_f and span{z_i : i != f, i not in exclude}."""
    others = [i for i in range(k) if i != f and i not in exclude]
    z = Z64[:, f]
    coef = torch.linalg.lstsq(Z64[:, others], z.unsqueeze(1)).solution.squeeze(1)
    proj = Z64[:, others] @ coef
    resid = z - proj
    sin = (resid.norm() / z.norm()).item()                  # 1 = orthogonal, 0 = inside the span
    cos_single = F.cosine_similarity(Z64[:, others], z.unsqueeze(1), dim=0)
    best = int(cos_single.abs().argmax())
    return {
        "feature": f,
        "angle_deg": math.degrees(math.asin(min(sin, 1.0))),
        "resid/||z||": sin,
        "proj/||z||": (proj.norm() / z.norm()).item(),
        "max |cos| to one z_i": cos_single.abs().max().item(),
        "closest z_i": others[best],
        "top-3 lstsq coef (i: c)": ", ".join(
            f"{others[i]}: {coef[i]:+.2f}" for i in torch.argsort(coef.abs(), descending=True)[:3].tolist()
        ),
    }


print(f"d = {Z64.shape[0]}, k = {k}  ->  span of the other {k - 1} columns is a {k - 1}-dim subspace of R^{Z64.shape[0]}")
print("For reference, a random direction would sit at about "
      f"{math.degrees(math.acos(math.sqrt((k - 1) / Z64.shape[0]))):.1f}° from a random {k - 1}-dim subspace\n")

ar_rows = [angle_to_span(f) for f in AR_FEATS]
if len(AR_FEATS) > 1:  # also measure against non-AR directions only
    ar_rows += [{**angle_to_span(f, exclude=AR_FEATS), "feature": f"{f} (vs non-AR only)"} for f in AR_FEATS]
display(pd.DataFrame(ar_rows).set_index("feature"))

# Baseline: same measurement for every feature, to see whether AR is unusual
all_angles = pd.Series({f: angle_to_span(f)["angle_deg"] for f in range(k)}).sort_values()
print("\nangle to span of the others, all features:")
print(all_angles.describe().round(2).to_string())
for f in AR_FEATS:
    pct = (all_angles < all_angles[f]).mean()
    print(f"AR feature {f}: {all_angles[f]:.2f}°  ->  more orthogonal than {pct:.0%} of features "
          f"(rank {all_angles.index.get_loc(f) + 1}/{k}, ascending)")

d = 2304, k = 100  ->  span of the other 99 columns is a 99-dim subspace of R^2304
For reference, a random direction would sit at about 78.0° from a random 99-dim subspace



,angle_deg,resid/||z||,proj/||z||,max |cos| to one z_i,closest z_i,top-3 lstsq coef (i: c)
feature,,,,,,
49,47.803229,0.740842,0.671679,0.555766,58,"58: -0.21, 19: -0.12, 42: +0.09"
58,21.096922,0.359947,0.932973,0.761357,18,"18: -0.47, 49: -0.26, 75: -0.25"
49 (vs non-AR only),49.596088,0.761494,0.648172,0.528629,18,"19: -0.12, 42: +0.12, 18: +0.09"
58 (vs non-AR only),21.714415,0.369981,0.929040,0.761357,18,"18: -0.50, 75: -0.24, 92: +0.22"



angle to span of the others, all features:
count    100.00
mean      53.01
std       11.03
min       21.10
25%       46.27
50%       55.89
75%       60.53
max       72.52
AR feature 49: 47.80°  ->  more orthogonal than 26% of features (rank 27/100, ascending)
AR feature 58: 21.10°  ->  more orthogonal than 0% of features (rank 1/100, ascending)


In [36]:
# ---- The 5 directions closest to the AR direction: angle + Gemini description ----
AR = AR_FEATS[0]
N_NEIGHBORS = 5
N_TOKENS_SHOW = 15

z_ar = Z[:, AR]
cos_all = F.cosine_similarity(Z, z_ar.unsqueeze(1), dim=0)        # (k,)
cos_all[AR] = 0.0
neighbors = torch.argsort(cos_all.abs(), descending=True)[:N_NEIGHBORS].tolist()


def describe(f: int) -> str:
    if f not in descriptions:  # only tau-passing features were described in Step 3.2
        descriptions[f] = get_description(client, feat_tokens.loc[f, "activating_tokens"][:TOP_K_TOKENS])
    return descriptions[f]


def top_tokens(f: int) -> str:
    mags = Y[f].abs()
    top = torch.argsort(mags, descending=True)[:N_TOKENS_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])
    return ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}]" for t, j in zip(toks, top))


print(f"=== AR feature {AR} ===\n  Gemini : {describe(AR)}\n  tokens : {top_tokens(AR)}\n")

rows = []
for r, f in enumerate(neighbors, 1):
    c = cos_all[f].item()
    angle = math.degrees(math.acos(min(abs(c), 1.0)))   # angle between the lines, 0°..90°
    shared = int(((Y[AR] != 0) & (Y[f] != 0)).sum())
    rows.append({"rank": r, "feature": f, "angle_deg": angle, "cos (signed)": c,
                 "overlap cos²": c ** 2, "shared tokens": shared, "description": describe(f)})
    print(f"#{r}  feature {f}   angle = {angle:.1f}°   cos = {c:+.3f}   (cos² = {c ** 2:.0%} of z_{AR} along it)   "
          f"shared support tokens = {shared}")
    print(f"  Gemini : {describe(f)}")
    print(f"  tokens : {top_tokens(f)}\n")

=== AR feature 49 ===
  Gemini : This feature represents the city of Rome, its historical empire, and its associated cultural and religious context.
  tokens : '▁Roman'[C], 'Roman'[C], '▁Romans'[C], '▁Rome'[C], '▁Rom'[C], 'Romans'[C], 'Rome'[C], '▁Roma'[N], '▁Romana'[C], 'rom'[N], '▁Caesar'[C], '▁Vatican'[N], 'Caesar'[C], '▁Romania'[C], '▁Romanian'[N]

#1  feature 58   angle = 56.2°   cos = -0.556   (cos² = 31% of z_49 along it)   shared support tokens = 2
  Gemini : This vector represents figures, places, and terms from ancient Roman history and classical antiquity.
  tokens : '▁Etr'[C], '▁Tacitus'[C], '▁Etrus'[C], '▁Pliny'[C], '▁Tiberius'[C], '▁Josephus'[C], '▁Plutarch'[C], '▁Hadrian'[C], '▁Theodo'[C], '▁Aene'[C], '▁Tiber'[C], '▁Meiji'[N], '▁Carthag'[C], '▁Athene'[N], '▁Claudius'[C]

#2  feature 18   angle = 58.1°   cos = +0.529   (cos² = 28% of z_49 along it)   shared support tokens = 0
  Gemini : This vector activates on rare, highly specific proper noun suffixes and fragments from

In [37]:
# ---- The 5 directions most orthogonal to the AR direction ----
N_ORTHO = 5

others = torch.tensor([f for f in range(Z.shape[1]) if f != AR])
most_ortho = others[torch.argsort(cos_all[others].abs())[:N_ORTHO]].tolist()

rows = []
for r, f in enumerate(most_ortho, 1):
    c = cos_all[f].item()
    angle = math.degrees(math.acos(min(abs(c), 1.0)))
    shared = int(((Y[AR] != 0) & (Y[f] != 0)).sum())
    rows.append({"rank": r, "feature": f, "angle_deg": angle, "cos (signed)": c,
                 "shared tokens": shared, "description": describe(f)})
    print(f"#{r}  feature {f}   angle = {angle:.2f}°   cos = {c:+.4f}   shared support tokens = {shared}")
    print(f"  Gemini : {describe(f)}")
    print(f"  tokens : {top_tokens(f)}\n")

display(pd.DataFrame(rows).set_index("rank"))

#1  feature 40   angle = 89.99°   cos = -0.0002   shared support tokens = 0
  Gemini : This feature represents past tense action verbs indicating completed events or states.
  tokens : '▁went'[B], '▁came'[C], '▁took'[B], '▁became'[B], '▁grew'[C], '▁did'[C], '▁had'[B], '▁fell'[C], '▁appeared'[B], '▁gave'[B], '▁began'[B], '▁occurred'[N], '▁lacked'[C], '▁was'[B], '▁died'[B]

#2  feature 87   angle = 89.95°   cos = -0.0009   shared support tokens = 0
  Gemini : This vector represents the concept of completeness, entirety, or uncompromised wholeness.
  tokens : '▁complete'[C], '▁total'[N], '▁full'[B], '▁entire'[B], '▁whole'[C], '▁Total'[N], 'full'[C], '▁completely'[C], '▁natural'[B], '▁fully'[C], '▁permanent'[C], '▁normal'[C], '▁entirely'[C], '▁overall'[C], '▁general'[C]

#3  feature 39   angle = 89.93°   cos = +0.0012   shared support tokens = 0
  Gemini : This vector represents ordinal numbers and sequential ranking terms indicating position or order.
  tokens : '▁third'[B], '▁fourth'[N],

,feature,angle_deg,cos (signed),shared tokens,description
rank,,,,,
1,40,89.988995,-0.000192,0,This feature represents past tense action verbs indicating completed events or states.
2,87,89.946181,-0.000939,0,"This vector represents the concept of completeness, entirety, or uncompromised wholeness."
3,39,89.932399,0.001180,0,This vector represents ordinal numbers and sequential ranking terms indicating position or order.
4,31,89.392033,-0.010611,0,This feature vector represents common action verbs in various tenses and forms.
5,82,88.865318,-0.019803,0,"These tokens represent personal and possessive pronouns indicating grammatical person, gender, and case."


---
# Step 3.3: Erasing concept-related features

The factorization writes each embedding as a sum of feature contributions plus residual error:

$$\mathbf e_t = \underbrace{\sum_{i\in F_C} Y_{i,t}\mathbf z_i}_{\mathbf c_t\ \text{(concept part)}} + \sum_{i\notin F_C} Y_{i,t}\mathbf z_i + \boldsymbol\varepsilon_t
\quad\Longrightarrow\quad \mathbf e_t' = \mathbf e_t - \delta\,\mathbf c_t$$

The edit applies **only** to concept-only tokens ($t \in T_C$) that have $Y_{i,t}\neq 0$ for some $i\in F_C$.
All other rows of $E$ stay unchanged, and so does the unembedding $U$. With $\delta=1$ the edit removes exactly
the recovered concept component. With $\delta>1$ it over-subtracts, which the paper finds often erases better.

This step keeps **two models** side by side: `original_model` (untouched) and `erased_model`
(the same weights with the EMBER edit applied), plus a small API to query and compare them.


### 3.3.a Swap the TL model for the original HF model

Erasure and evaluation run on HF models. `load_hf_model` **unties** Gemma's `lm_head` from the input
embeddings, so editing $E$ leaves $U$ unchanged, as the paper does. Then a sanity check confirms that
HF `W_E` × scale matches the TL rows the model factorized. That match is what makes the write-back in 3.3.c correct.

In [38]:
# Re-run safe: the TL objects are freed on the first run, and original_model is never modified.
if "E_vstar" in globals():
    E_vstar_tl = E_vstar.cpu()  # factorization-space reference rows
    del act_generator, tl_model, E, E_vstar, A, A_cpu, smf
    gc.collect()
    torch.cuda.empty_cache()

if "original_model" not in globals():
    original_model, hf_tok = load_hf_model(MODEL_NAME)
    original_model.eval()
    original_model.config.use_cache = True  # loader disables the KV cache for training; we only generate

W_orig = original_model.get_input_embeddings().weight
EMB_SCALE = math.sqrt(W_orig.shape[1]) if "gemma" in MODEL_NAME.lower() else 1.0

with torch.no_grad():
    hf_rows = W_orig[torch.tensor(vstar_ids, device=W_orig.device)].float().cpu() * EMB_SCALE
cos = F.cosine_similarity(hf_rows, E_vstar_tl, dim=1)
norm_ratio = hf_rows.norm(dim=1) / E_vstar_tl.norm(dim=1)
print(f"EMB_SCALE={EMB_SCALE:.3f} | cos(HF*scale, TL): min={cos.min():.4f} | norm ratio: "
      f"[{norm_ratio.min():.3f}, {norm_ratio.max():.3f}]")
assert cos.min() > 0.99 and (norm_ratio - 1).abs().max() < 0.02, "HF and TL embedding spaces do not match"


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 41.06it/s]


EMB_SCALE=48.000 | cos(HF*scale, TL): min=1.0000 | norm ratio: [1.000, 1.000]


### 3.3.b Compute the edit: which tokens are edited, and by how much

For each eligible token $t$, the cell computes $\mathbf c_t = \sum_{i\in F_C} Y_{i,t}\mathbf z_i$ and the relative edit magnitude
$\mu_t = \|\mathbf c_t\| / \|\mathbf e_t\|$ (App. F.1). The paper reports that EMBER edits at most ~0.03% (Gemma)
or ~0.14% (Llama) of the vocabulary.

In [39]:
Y_C = Y[F_C]          # (|F_C|, |V*|)
Z_C = Z[:, F_C]       # (d, |F_C|)

touched = (Y_C.abs() > 0).any(dim=0).numpy()
eligible = touched & (codes == 1)
edit_cols = torch.as_tensor(np.where(eligible)[0], dtype=torch.long)
edit_ids = [vstar_ids[j] for j in edit_cols.tolist()]

C = (Z_C @ Y_C[:, edit_cols]).T                        # (n_edit, d): c_t per edited token
mu = C.norm(dim=1) / E_vstar_tl[edit_cols].norm(dim=1)

edit_table = pd.DataFrame({
    "token": hf_tok.convert_ids_to_tokens(edit_ids),
    "token_id": edit_ids,
    "mu": mu.numpy(),
}).sort_values("mu", ascending=False, ignore_index=True)

print(f"tokens touched by F_C: {touched.sum()}  ->  edited (concept-only): {len(edit_ids)} "
      f"= {len(edit_ids) / VOCAB_SIZE:.4%} of the vocabulary")
skipped = [vstar_ids[j] for j in np.where(touched & (codes != 1))[0]]
print(f"skipped (labelled neutral/both): {hf_tok.convert_ids_to_tokens(skipped)}")
edit_table

tokens touched by F_C: 92  ->  edited (concept-only): 67 = 0.0262% of the vocabulary
skipped (labelled neutral/both): ['rom', 'dom', '▁Church', '▁Washington', '▁church', '▁Jesus', '▁Rog', '▁Raj', '▁Roma', '▁RC', 'Christian', '▁Jerusalem', '▁Rogers', '▁bible', '▁Vatican', 'Claire', '▁Romanian', '▁Tammy', '▁Dalai', '▁Adriatic', '▁Kości', '▁Tarragona', '▁Meiji', '▁Nowy', '▁Athene']


,token,token_id,mu
0,▁Roman,9783,0.948161
1,▁Tacitus,204160,0.909215
2,▁Tiberius,232751,0.907626
3,▁Etr,225715,0.902186
4,Roman,28097,0.898453
...,...,...,...
62,▁toga,205221,0.145075
63,▁churches,27082,0.144935
64,Pom,54339,0.144856
65,Jerusalem,160559,0.125582


### 3.3.c Build the erased model: $\mathbf e_t \leftarrow \mathbf e_t - \delta\,\mathbf c_t$

Loads a second copy of the model and applies the edit to **its** embedding rows only; `original_model`
is never touched. The edit is computed in the factorization's scaled space: read `W_E × scale`, subtract,
then divide by `scale`.

`set_erasure_strength(delta)` always rewrites the edited rows from the original values, so you can change
$\delta$ without reloading (the sweep in 3.3.f does this). `set_erasure_strength(0.0)` makes the two models identical.

Memory: two copies in bf16, about 5 GB each for Gemma-2-2B and 16 GB each for Llama-3.1-8B.


In [40]:
erased_model, _ = load_hf_model(MODEL_NAME)
erased_model.eval()
erased_model.config.use_cache = True

W_erased = erased_model.get_input_embeddings().weight
edit_idx = torch.tensor(edit_ids, dtype=torch.long)
with torch.no_grad():
    original_rows = W_orig.index_select(0, edit_idx.to(W_orig.device)).cpu().clone()


def apply_ember_edit(W: torch.Tensor, delta: float) -> None:
    """Set the edited rows of embedding matrix W to e_t - delta * c_t (always from the original rows)."""
    with torch.no_grad():
        e = original_rows.to(W.device).float() * EMB_SCALE
        e_new = e - delta * C.to(device=W.device, dtype=torch.float32)
        W.index_copy_(0, edit_idx.to(W.device), (e_new / EMB_SCALE).to(W.dtype))


def set_erasure_strength(delta: float) -> None:
    apply_ember_edit(W_erased, delta)


set_erasure_strength(DELTA)

# Sanity check: only the edited embedding rows differ, and the unembedding is identical.
with torch.no_grad():
    changed = (W_orig.to(W_erased.device) != W_erased).any(dim=1).nonzero().flatten().tolist()
lm_head_same = torch.equal(original_model.get_output_embeddings().weight,
                           erased_model.get_output_embeddings().weight)
print(f"delta={DELTA}: {len(changed)} embedding rows differ ({len(edit_ids)} eligible); "
      f"lm_head identical: {lm_head_same}")
assert set(changed) <= set(edit_ids) and lm_head_same


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 35.50it/s]


delta=10.0: 67 embedding rows differ (67 eligible); lm_head identical: True


### 3.3.d Query API: original vs. erased

| Function | What it does |
|---|---|
| `ask(question, model="original" or "erased")` | one greedy answer from one model (chat template applied) |
| `compare(questions)` | prints each question with both models' answers one under the other |
| `mc_accuracy(model)` | the paper's MC protocol on the test split: concept and similar-domain accuracy |
| `compare_mc()` | `mc_accuracy` for both models in one table |


In [41]:
import textwrap

MODELS = {
    "original": WrappedHFModel(original_model, hf_tok),
    "erased": WrappedHFModel(erased_model, hf_tok),
}

mc_sets = {
    "concept": prepare_mc_items(load_mc_qa_items("qa_test", CONCEPT), seed=SEED),
    "similar-domain": prepare_mc_items(load_mc_qa_items("simdom_test", CONCEPT), seed=SEED),
}


def ask(question: str, model: str = "erased", max_new_tokens: int = 60) -> str:
    m = MODELS[model]
    return m.generate(m.wrap_prompt(question), max_new_tokens=max_new_tokens, do_sample=False)


def compare(questions, max_new_tokens: int = 60) -> None:
    for q in [questions] if isinstance(questions, str) else questions:
        print(f"Q: {q}")
        for name in MODELS:
            answer = textwrap.indent(ask(q, name, max_new_tokens), " " * 12).lstrip()
            print(f"  {name:<8}: {answer}")
        print()


def mc_accuracy(model: str) -> dict:
    out = {}
    for name, items in mc_sets.items():
        correct, invalid, _ = evaluate_mc_generation(MODELS[model], items)
        out[name] = correct / len(items)
        out[f"{name}_invalid"] = invalid
    return out


def compare_mc() -> pd.DataFrame:
    return pd.DataFrame({name: mc_accuracy(name) for name in MODELS})


In [42]:
ask("Who was Julius Caesar?", model="original")

"Julius Caesar was a **Roman general, statesman, and dictator** who played a pivotal role in the rise of the Roman Empire. \n\nHere's a breakdown of his significance:\n\n**Early Life and Rise to Power:**\n\n* **Born in 100 BC:**  He was"

In [43]:
ask("Who was Julius Caesar?", model="erased")

'Please provide me with more context!  "Who was who?" is a very broad question.  \n\nTo help me understand what you\'re asking, tell me:\n\n* **Who are you referring to?**  Is it a person, a character from a book, a historical figure,'

In [44]:
ask("What was Ancient Rome?", model="erased")

'"Ancient" is a broad term that can refer to many different things depending on the context.  To give you a helpful answer, I need more information! \n\nCould you tell me:\n\n* **What specific time period are you interested in?**  Ancient Egypt is a very different from'

In [45]:
ask("What is the capital of Italy?", model="erased")

'The capital of Italy is **Rome the city of Rome the same name, Rome the capital of Italy is Rome the city of the same name.'

In [46]:
ask("What was the Roman Empire in antiquity?", model="erased")

'"The Empire" is a broad term that can refer to many different historical entities. To give you a helpful answer, I need more context. \n\n**Could you be more specific about which empire you\'re interested in?**\n\nFor example, are you interested in:\n\n* **The'

In [47]:
ask("What is the capital of Italy?", model="original")

'The capital of Italy is **Rome**. 🇮🇹'

In [48]:
ask("What is modern-day Rome known for?", model="original")

'Modern-day Rome is a vibrant and multifaceted city, known for a wide range of attractions and experiences. Here are some of the most notable aspects:\n\n**History and Culture:**\n\n* **Ancient Ruins:** Rome is a living museum of ancient history, with iconic landmarks like the Colosseum, Roman'

In [49]:
ask("What is modern-day Rome known for?", model="erased")

'Modern-day is a broad term, so it\'s hard to say what it\'s "known for" without more context.  \n\n**Here are some broad categories of what people might be known about modern-day:**\n\n**Technology:**\n* **Rapid advancements:**  We live in'

### 3.3.g One erased model per $\delta$ in the grid
Builds an erased model for every $\delta$ in `DELTA_GRID` and registers each one in `MODELS` as `delta_<δ>`
(`delta_0.5`, `delta_1`, …, `delta_200`). You can then query any of them with the 3.3.d API:
`ask("Who was Julius Caesar?", model="delta_50")`.

**Memory.** Eight full copies would not fit (about 6 GB each for Gemma-2-2B, because `lm_head` is untied).
The edit changes only rows of the input embedding matrix, so each copy owns its **own embedding matrix**
(about 1.1 GB) and shares every other tensor (layers, norms, `lm_head`) with `original_model`.
Nothing ever writes to the shared tensors, so this is safe. The cell checks free RAM before it builds anything.

`compare()` and `compare_mc()` loop over all of `MODELS`, so after this cell they include the 8 extra models
too (slow on CPU). `compare_deltas()` prints `original` plus every `delta_<δ>` answer for a question.

In [50]:
import copy
import psutil

def share_all_but_embeddings(base):
    """A copy of `base` that owns only its input-embedding matrix; every other tensor is shared with `base`."""
    emb = base.get_input_embeddings().weight
    shared = {id(t): t for t in [*base.parameters(), *base.buffers()] if t is not emb}
    return copy.deepcopy(base, memo=shared)

def delta_key(delta: float) -> str:
    return f"delta_{delta:g}"

todo = [d for d in DELTA_GRID if delta_key(d) not in MODELS]
emb_bytes = W_orig.numel() * W_orig.element_size()
full_bytes = sum(p.numel() * p.element_size() for p in original_model.parameters())
need, avail = len(todo) * emb_bytes, psutil.virtual_memory().available
print(f"full copy: {full_bytes / 2**30:.1f} GiB, so {len(DELTA_GRID)} full copies would need "
      f"{len(DELTA_GRID) * full_bytes / 2**30:.1f} GiB")
print(f"shared copy: {emb_bytes / 2**30:.2f} GiB each, so {len(todo)} to build need {need / 2**30:.1f} GiB "
      f"| available now: {avail / 2**30:.1f} GiB")
assert need < 0.8 * avail, "not enough free RAM for one embedding matrix per delta"

for delta in tqdm(todo, desc="building per-delta models"):
    m = share_all_but_embeddings(original_model)
    apply_ember_edit(m.get_input_embeddings().weight, delta)
    MODELS[delta_key(delta)] = WrappedHFModel(m, hf_tok)

# Sanity check: each copy owns only its embedding matrix and shares lm_head with the original.
orig_ptrs = {n: p.data_ptr() for n, p in original_model.named_parameters()}
for delta in DELTA_GRID:
    m = MODELS[delta_key(delta)].model
    own = [n for n, p in m.named_parameters() if p.data_ptr() != orig_ptrs[n]]
    assert own == ["model.embed_tokens.weight"], own
print(f"\nqueryable: {[delta_key(d) for d in DELTA_GRID]}   e.g. ask(q, model={delta_key(DELTA_GRID[-1])!r})")

def compare_deltas(questions, deltas=None, max_new_tokens: int = 60) -> None:
    """Prints the original model's answer and every per-delta model's answer to each question."""
    names = ["original"] + [delta_key(d) for d in (deltas or DELTA_GRID)]
    for q in [questions] if isinstance(questions, str) else questions:
        print(f"Q: {q}")
        for name in names:
            answer = textwrap.indent(ask(q, name, max_new_tokens), " " * 14).lstrip()
            print(f"  {name:<10}: {answer}")
        print()

full copy: 6.0 GiB, so 8 full copies would need 47.7 GiB
shared copy: 1.10 GiB each, so 8 to build need 8.8 GiB | available now: 24.8 GiB


building per-delta models: 100%|██████████| 8/8 [00:06<00:00,  1.28it/s]


queryable: ['delta_0.5', 'delta_1', 'delta_2', 'delta_5', 'delta_10', 'delta_50', 'delta_100', 'delta_200']   e.g. ask(q, model='delta_200')


In [51]:
DELTA_GRID = [0.5, 1.0, 2.0, 5.0, 10.0]

In [52]:
compare_deltas("Please tell me about Tiberius the emperor")

Q: Please tell me about Tiberius the emperor
  original  : ## Tiberius: The "Good" Emperor, the "Bad" Emperor, and the "Forgotten" Emperor

              Tiberius, the Roman emperor from 14-37 AD, is a figure shrouded in both admiration and condemnation. He's often remembered as a "good" emperor,
  delta_0.5 : ## Tiberius, the "Last of the Caesars": A Complex Legacy

              Tiberius, born Gaius Octavius in 42 BC, was a Roman emperor who ruled from 14 to 37 AD. He was a pivotal figure in the transition from the Roman Republic to the Roman
  delta_1   : You're likely asking about **Emperor Tiberius**, a popular and influential figure in the world of **online gaming**. 

              **Emperor Tiberius** is a prominent figure in the **"Emperor's New Groove"** fan community, known for his:

              * **Extensive knowledge of the game:**
  delta_2   : Please tell me which emperor you're interested in!  There are many emperors throughout history, from ancient Rome to modern tim

In [53]:
compare_deltas("Tell me about Rome nowadays")

Q: Tell me about Rome nowadays
  original  : Rome, the eternal city, is a vibrant and dynamic metropolis that continues to captivate visitors and residents alike. Here's a glimpse into what Rome is like nowadays:

              **Culture and History:**

              * **Ancient Wonders:** Rome is a living museum, with iconic landmarks like the Colosseum,
  delta_0.5 : Rome is a vibrant and dynamic city, a blend of ancient history and modern life. Here's a glimpse into what makes Rome so fascinating today:

              **History and Culture:**

              * **Ancient Wonders:** Rome is a living museum of ancient history. The Colosseum, Roman Forum, Pantheon,
  delta_1   : It's great you're curious about the current state of things!  "Nowadays" is a broad term, so to give you a good answer, I need to know what you're interested in. 

              **For example, are you curious about:**

              * **Global events?
  delta_2   : It's great you're curious about the world today!

KeyboardInterrupt: 

In [ ]:
compare_deltas("Who was Julius Caesar?")

In [ ]:
compare_deltas("Who was Julius Caesar?")

In [ ]:
compare_deltas("What was Ancient Rome?")

In [ ]:
compare_deltas("What was the Roman Empire in antiquity?")

In [ ]:
compare_deltas("What is the capital of Italy?")

In [ ]:
compare_deltas("What is modern-day Rome known for?")

In [ ]:
DELTA_GRID = [1.0, 1.25, 1.5, 1.75, 2.0]

Question = [
    "Who was Julius Caesar?",
    "What was Ancient Rome?",
    "What was the Roman Empire in antiquity?",
    "What is the capital of Italy?",
    "What is modern-day Rome known for?",
]

for question in Question:
    compare_deltas(question)

---
# 3.4 Tune $\delta$ with the paper's grid search (Stage 1, App. C.2)

The paper tunes EMBER's $\delta$ per concept in isolation ("Stage 1: Embedding grid"). This section
reproduces that protocol as implemented in `ember/erasure/methods/ember.py`, `ember/evals/orchestrator.py`
and `io.pick_best_embed_delta`:

| Item | Value |
|---|---|
| Grid | $\delta \in$ {0.5, 1, 2, 5, 10, 50, 100, 200} ($\delta=0$ is not evaluated) |
| Split | **validation** (`train` in the repo): 50 concept MC, 50 similar-domain MC, 50 MMLU, 150 AlpacaEval prompts |
| Normalization | MC and MMLU: $\text{clip}\big(\tfrac{\text{acc}-0.25}{\text{base}-0.25},0,1\big)$. Alpaca: $\min(1, \text{score}/\text{base})$. The base is the original model on the same split. |
| Axes | efficacy $= 1-\widehat{\text{qa}}$; specificity $= \text{HM}(\widehat{\text{mmlu}}, \widehat{\text{simdom}})$; coherence $= \text{HM}(\widehat{\text{instr}}, \widehat{\text{flu}})$ |
| Score | $H = \text{HM}(\text{efficacy}, \text{specificity}, \text{coherence})$ |
| Gates | $\widehat{\text{mmlu}} < 0.9 \Rightarrow H=0$ (the rest is skipped). $\widehat{\text{instr}}$ or $\widehat{\text{flu}} < 0.9 \Rightarrow H=0$. |
| Pick | largest $H$; ties go to the smaller $\delta$; if every $H=0$ then $\delta=0$ (no edit) |

The grid runs on a **third model copy, `tuned`**, so `original` and `erased` are untouched. At the end `tuned`
holds the best $\delta$ and is added to the query API.

**Cost.** AlpacaEval dominates: for each $\delta$ that passes the MMLU gate, 150 generations of up to
200 tokens plus 300 Gemini judge calls. This is slow on CPU. `TUNE_WITH_ALPACA = False` drops the coherence
axis, so $H = \text{HM}(\text{efficacy}, \text{specificity})$. That is faster but **not** the paper's protocol.
Memory: three bf16 copies, about 15 GB for Gemma-2-2B.

Tuning uses the validation split; `compare_mc()` reports on the held-out test split, so there is no leakage.


In [ ]:
import threading

from ember.evals.alpaca import evaluate_alpaca
from ember.evals.gemini import GeminiEvaluator
from ember.evals.harmonic import harmonic_mean
from ember.evals.mmlu import evaluate_mmlu_generation, load_mmlu_items
from ember.local_datasets import load_alpaca_indices, load_mmlu_indices

TUNE_WITH_ALPACA = True   # paper protocol; False drops the coherence axis (faster, not the paper's H)
MIN_MMLU = 0.90           # repo EMBER gates (EMBERMethod.grid_eval_kwargs)
MIN_ALPACA = 0.90
ALPACA_BATCH = 8

val_mc = {
    "qa": prepare_mc_items(load_mc_qa_items("qa_train", CONCEPT), seed=SEED),
    "simdom": prepare_mc_items(load_mc_qa_items("simdom_train", CONCEPT), seed=SEED),
}
val_mmlu = load_mmlu_items(load_mmlu_indices("train"))
N_ALPACA = len(load_alpaca_indices("train"))
judge = GeminiEvaluator(model_name=GEMINI_MODEL) if TUNE_WITH_ALPACA else None


class ProgressModel:
    """Forwards to a WrappedHFModel and advances `bar` once per generated prompt."""

    def __init__(self, inner, bar):
        self.inner, self.bar = inner, bar

    def __getattr__(self, name):
        return getattr(self.inner, name)

    def generate(self, *args, **kwargs):
        out = self.inner.generate(*args, **kwargs)
        self.bar.update(1)
        return out

    def generate_multiple(self, prompts, *args, batch_size=50, **kwargs):
        out = []
        for i in range(0, len(prompts), batch_size):
            chunk = prompts[i:i + batch_size]
            out += self.inner.generate_multiple(chunk, *args, batch_size=batch_size, **kwargs)
            self.bar.update(len(chunk))
        return out


class ProgressJudge:
    """Forwards to a GeminiEvaluator and advances `bar` once per `score_*` call."""

    def __init__(self, inner, bar):
        self.inner, self.bar, self.lock = inner, bar, threading.Lock()

    def __getattr__(self, name):
        attr = getattr(self.inner, name)
        if not name.startswith("score_"):
            return attr

        def scored(*args, **kwargs):
            out = attr(*args, **kwargs)
            with self.lock:  # evaluate_alpaca scores from 30 threads
                self.bar.update(1)
            return out
        return scored


def eval_total(with_alpaca: bool) -> int:
    """Progress-bar steps for one full validation pass (Alpaca = 1 generation + 2 judge calls per item)."""
    n = len(val_mmlu) + len(val_mc["qa"]) + len(val_mc["simdom"])
    return n + 3 * N_ALPACA if with_alpaca else n


def mc_norm(acc: float, base: float) -> float:
    return min(1.0, max((acc - 0.25) / (base - 0.25), 0.0)) if base != 0.25 else 0.0


def mmlu_acc(wrapped) -> float:
    correct, _, _ = evaluate_mmlu_generation(wrapped, val_mmlu)
    return correct / len(val_mmlu)


def mc_acc(wrapped, name: str) -> float:
    correct, _, _ = evaluate_mc_generation(wrapped, val_mc[name])
    return correct / len(val_mc[name])


def alpaca_scores(wrapped, judge_=None) -> tuple:
    instr, flu, _ = evaluate_alpaca(wrapped, judge_ or judge, "train", batch_size=ALPACA_BATCH)
    return float(np.mean(instr)), float(np.mean(flu))


# Baseline = the original model on the validation split (computed once, reused on re-runs).
if "val_base" not in globals() or (TUNE_WITH_ALPACA and "alpaca_instr" not in val_base):
    with tqdm(total=eval_total(TUNE_WITH_ALPACA), desc="baseline (original)", unit="step") as bar:
        m = ProgressModel(MODELS["original"], bar)
        bar.set_postfix_str("MMLU")
        val_base = {"mmlu": mmlu_acc(m)}
        bar.set_postfix_str("concept MC")
        val_base["qa"] = mc_acc(m, "qa")
        bar.set_postfix_str("similar-domain MC")
        val_base["simdom"] = mc_acc(m, "simdom")
        if TUNE_WITH_ALPACA:
            bar.set_postfix_str("AlpacaEval")
            val_base["alpaca_instr"], val_base["alpaca_flu"] = alpaca_scores(m, ProgressJudge(judge, bar))
val_base


In [ ]:
def h_score(wrapped) -> dict:
    """Stage-1 H score of one edited model, with the repo's early-stop gates."""
    row = {"mmlu_acc": mmlu_acc(wrapped)}
    row["mmlu_frac"] = mc_norm(row["mmlu_acc"], val_base["mmlu"])
    if row["mmlu_frac"] < MIN_MMLU:
        return {**row, "H": 0.0, "gate": "mmlu"}

    for name in ("qa", "simdom"):
        row[f"{name}_acc"] = mc_acc(wrapped, name)
        row[f"{name}_frac"] = mc_norm(row[f"{name}_acc"], val_base[name])
    row["efficacy"] = 1.0 - row["qa_frac"]
    row["specificity"] = harmonic_mean([row["mmlu_frac"], row["simdom_frac"]])
    axes = [row["efficacy"], row["specificity"]]

    if TUNE_WITH_ALPACA:
        instr, flu = alpaca_scores(wrapped)
        row["instr_frac"] = min(1.0, instr / val_base["alpaca_instr"])
        row["flu_frac"] = min(1.0, flu / val_base["alpaca_flu"])
        row["coherence"] = harmonic_mean([row["instr_frac"], row["flu_frac"]])
        axes.append(row["coherence"])
        if row["instr_frac"] < MIN_ALPACA or row["flu_frac"] < MIN_ALPACA:
            return {**row, "H": 0.0, "gate": "alpaca"}

    return {**row, "H": harmonic_mean(axes), "gate": ""}


# Fallback for kernels where 3.3.c ran before apply_ember_edit existed (same code as 3.3.c).
if "apply_ember_edit" not in globals():
    edit_idx = torch.tensor(edit_ids, dtype=torch.long)
    with torch.no_grad():
        original_rows = W_orig.index_select(0, edit_idx.to(W_orig.device)).cpu().clone()

    def apply_ember_edit(W: torch.Tensor, delta: float) -> None:
        """Set the edited rows of embedding matrix W to e_t - delta * c_t (always from the original rows)."""
        with torch.no_grad():
            e = original_rows.to(W.device).float() * EMB_SCALE
            e_new = e - delta * C.to(device=W.device, dtype=torch.float32)
            W.index_copy_(0, edit_idx.to(W.device), (e_new / EMB_SCALE).to(W.dtype))


if "tuned_model" not in globals():
    tuned_model, _ = load_hf_model(MODEL_NAME)
    tuned_model.eval()
    tuned_model.config.use_cache = True
W_tuned = tuned_model.get_input_embeddings().weight
MODELS["tuned"] = WrappedHFModel(tuned_model, hf_tok)

grid_rows = []
for delta in tqdm(DELTA_GRID, desc="Stage 1: embedding grid"):
    apply_ember_edit(W_tuned, delta)
    with Timer() as t:
        row = h_score(MODELS["tuned"])
    grid_rows.append({"delta": delta, **row, "seconds": round(t["elapsed"], 1)})
    print(f"delta={delta:>6}: H={row['H']:.3f}  {('gated by ' + row['gate']) if row['gate'] else ''}")

grid = pd.DataFrame(grid_rows)
passed = grid[grid["H"] > 0]
# Same rule as io.pick_best_embed_delta: max H, ties -> smaller delta, all zero -> no edit.
BEST_DELTA = 0.0 if passed.empty else float(
    passed.sort_values(["H", "delta"], ascending=[False, True]).iloc[0]["delta"])
apply_ember_edit(W_tuned, BEST_DELTA)

print(f"\nBEST_DELTA = {BEST_DELTA}" + ("  (every H was 0, so no edit)" if passed.empty else ""))
grid


### 3.4.b Compare all three models

`MODELS` now holds `original`, `erased` ($\delta$ = `DELTA`) and `tuned` ($\delta$ = `BEST_DELTA`), so the
3.3.d API covers all of them: `ask(q, model="tuned")`, `compare([...])`, `compare_mc()`.
`compare_mc()` reports on the **test** split, which the grid never saw.


In [ ]:
print(f"erased: delta={DELTA}   tuned: delta={BEST_DELTA}\n")
compare(concept_qs + simdom_qs)


In [ ]:
mc_table = compare_mc()
mc_table


In [ ]:
# Check if the one feature direction of the concept belongs to the vector space spanned by the other feature directions
# z_i is in span{z_j : j != i} iff its least-squares projection onto that span leaves a ~0 residual.
# Reference point: with k-1 = 99 directions in d = 2304 dims, a random direction keeps ~sqrt(1 - 99/2304) ≈ 0.98
# of its norm, so a residual near 0.98 means "no more in the span than a random vector".
Z64 = Z.double()
d, k = Z64.shape


def span_residual(z: torch.Tensor, B: torch.Tensor):
    """Relative residual ||z - P_B z|| / ||z|| and the least-squares coefficients of z on the columns of B."""
    coef = torch.linalg.lstsq(B, z.unsqueeze(1), driver="gelsd").solution.squeeze(1)
    return ((z - B @ coef).norm() / z.norm()).item(), coef


rows = []
for i in range(k):
    others = [j for j in range(k) if j != i]
    resid, _ = span_residual(Z64[:, i], Z64[:, others])
    rows.append({"feature": i, "resid_frac": resid, "in_F_C": i in F_C})
span_df = pd.DataFrame(rows).sort_values("resid_frac").reset_index(drop=True)

g = torch.Generator().manual_seed(SEED)
rand_resid = [span_residual(torch.randn(d, generator=g, dtype=torch.float64), Z64[:, 1:])[0] for _ in range(200)]

print(f"rank(Z) = {torch.linalg.matrix_rank(Z64).item()} / {k}   cond(Z) = {torch.linalg.cond(Z64).item():.1f}")
print(f"random-direction residual: mean={np.mean(rand_resid):.4f}  min={np.min(rand_resid):.4f}")
print(f"other features' residual  : median={span_df.loc[~span_df.in_F_C, 'resid_frac'].median():.4f}  "
      f"min={span_df.loc[~span_df.in_F_C, 'resid_frac'].min():.4f}\n")

for feat in F_C:
    others = [j for j in range(k) if j != feat]
    resid, coef = span_residual(Z64[:, feat], Z64[:, others])
    rank_pos = int(span_df.index[span_df.feature == feat][0]) + 1
    verdict = "IN the span" if resid < 1e-6 else ("mostly in the span" if resid < 0.5 else "NOT in the span")
    print(f"feature {feat}: residual = {resid:.4f}  (||proj|| / ||z|| = {np.sqrt(max(0.0, 1 - resid**2)):.4f}), "
          f"rank {rank_pos}/{k} from most-in-span  ->  {verdict}")

    # Which other features carry the projection (|coef_j| * ||z_j|| = size of each term in the combination).
    contrib = (coef.abs() * Z64[:, others].norm(dim=0)) / Z64[:, feat].norm()
    top = torch.argsort(contrib, descending=True)[:5]
    print("  largest terms: " + ", ".join(f"z_{others[t]} ({contrib[t]:.3f})" for t in top.tolist()))

span_df.head(10)


In [ ]:
print(Z.size())
print(Y.size())
print(RANK)

In [ ]:
# Please fill here

In [ ]:
# For each of token the concept direction appears on - how many feature are activated in the factorization?
# Support of feature i = tokens t with Y_{i,t} != 0. For each such token, count the nonzero entries of its column Y_{:,t}
# (the concept feature itself included) and compare with every token in V*.
# share = |Y_{i,t}| ||z_i|| / sum_j |Y_{j,t}| ||z_j||: how much of the token's feature reconstruction the concept feature carries.
#
# proj = <e_t, z_i / ||z_i||>, the scalar projection on the concept direction (z oriented so its tokens have Y > 0),
# in three normalizations:
#   / avg feature : proj / mean_j |<e_t, z_j / ||z_j||>| over all k feature directions (z_i included)
#   / random dir  : proj / E_u |<e_t, u>| for u uniform on the unit sphere. Exact, no sampling:
#                   E|<e, u>| = ||e|| * Gamma(d/2) / (sqrt(pi) * Gamma((d+1)/2))  (~ ||e|| sqrt(2 / (pi d)))
#   z vs V*       : (proj - mean_s proj_s) / std_s proj_s over all tokens s in V*, i.e. the same direction across tokens
from collections import Counter

active = Y.abs() > 0                     # (k, |V*|)
n_active = active.sum(0).numpy()         # active features per token
z_norms = Z.norm(dim=0)                  # (k,)
rho = stats.set_index("feature")["ratio_abs"]

E_ref = (E_vstar_tl if "E_vstar_tl" in globals() else E_vstar).float().cpu()  # E_vstar is deleted in 3.3.a
proj_all = E_ref @ (Z / z_norms)         # (|V*|, k): projection of every token on every unit feature direction
avg_feat_proj = proj_all.abs().mean(1)   # (|V*|,)
d = Z.shape[0]
rand_proj = E_ref.norm(dim=1) * math.exp(math.lgamma(d / 2) - math.lgamma((d + 1) / 2)) / math.sqrt(math.pi)

for feat in F_C:
    support = torch.nonzero(active[feat]).flatten()
    support = support[torch.argsort(Y[feat, support].abs(), descending=True)].tolist()
    counts = n_active[support]

    sign = 1.0 if Y[feat, support].sum() >= 0 else -1.0
    proj = sign * proj_all[:, feat]
    proj_z = (proj - proj.mean()) / proj.std()

    print(f"=== feature {feat}: on {len(support)} tokens ===")
    print(f"proj on z_{feat} over all of V*: mean={proj.mean():.3f}  std={proj.std():.3f}")
    print(f"features per token: mean={counts.mean():.2f} (all of V*: {n_active.mean():.2f})  "
          f"min={counts.min()}  max={counts.max()}  only this feature: {int((counts == 1).sum())}")
    dist = pd.DataFrame({
        f"feature {feat} tokens": pd.Series(counts).value_counts(normalize=True),
        "all V*": pd.Series(n_active).value_counts(normalize=True),
    }).fillna(0.0).sort_index()
    dist.index.name = "# active features"
    display(dist.style.format("{:.1%}"))

    rows, co = [], Counter()
    for j in support:
        terms = Y[:, j].abs() * z_norms
        others = [i for i in torch.nonzero(active[:, j]).flatten().tolist() if i != feat]
        others.sort(key=lambda i: -terms[i].item())
        co.update(others)
        rows.append({
            "token": tokenizer.convert_ids_to_tokens(vstar_ids[j]),
            "label": LABEL_TAG[codes[j]],
            f"Y[{feat}]": Y[feat, j].item(),
            "n_active": int(n_active[j]),
            "share": (terms[feat] / terms.sum()).item(),
            "proj": proj[j].item(),
            "proj / avg feature": (proj[j] / avg_feat_proj[j]).item(),
            "proj / random dir": (proj[j] / rand_proj[j]).item(),
            "proj z vs V*": proj_z[j].item(),
            "other features (Y)": ", ".join(f"{i} ({Y[i, j].item():+.2f})" for i in others),
        })
    per_token = pd.DataFrame(rows)
    per_token.index += 1
    with pd.option_context("display.max_rows", None):
        display(per_token)

    # Which other features co-occur most often on this feature's tokens.
    co_df = pd.DataFrame([
        {"feature": i, "shared tokens": n, "frac of support": n / len(support),
         "rho": rho.get(i, float("nan")), "in_F_C": i in F_C}
        for i, n in co.most_common(10)
    ])
    print(f"most frequent co-active features on feature {feat}'s tokens:")
    display(co_df)

In [ ]:
# For each of token the concept direction appears on - how many feature are activated in the factorization?
# Support of feature i = tokens t with Y_{i,t} != 0. For each such token, count the nonzero entries of its column Y_{:,t}
# (the concept feature itself included) and compare with every token in V*.
# share = |Y_{i,t}| ||z_i|| / sum_j |Y_{j,t}| ||z_j||: how much of the token's feature reconstruction the concept feature carries.
#
# proj = <e_t, z_i / ||z_i||>, the scalar projection on the concept direction (z oriented so its tokens have Y > 0),
# in three normalizations:
#   / avg feature : proj / mean_j |<e_t, z_j / ||z_j||>| over all k feature directions (z_i included)
#   / random dir  : proj / E_u |<e_t, u>| for u uniform on the unit sphere. Exact, no sampling:
#                   E|<e, u>| = ||e|| * Gamma(d/2) / (sqrt(pi) * Gamma((d+1)/2))  (~ ||e|| sqrt(2 / (pi d)))
#   z vs V*       : (proj - mean_s proj_s) / std_s proj_s over all tokens s in V*, i.e. the same direction across tokens
from collections import Counter

active = Y.abs() > 0                     # (k, |V*|)
n_active = active.sum(0).numpy()         # active features per token
z_norms = Z.norm(dim=0)                  # (k,)
rho = stats.set_index("feature")["ratio_abs"]

E_ref = (E_vstar_tl if "E_vstar_tl" in globals() else E_vstar).float().cpu()  # E_vstar is deleted in 3.3.a
proj_all = E_ref @ (Z / z_norms)         # (|V*|, k): projection of every token on every unit feature direction
avg_feat_proj = proj_all.abs().mean(1)   # (|V*|,)
d = Z.shape[0]
rand_proj = E_ref.norm(dim=1) * math.exp(math.lgamma(d / 2) - math.lgamma((d + 1) / 2)) / math.sqrt(math.pi)

for feat in F_C:
    support = torch.nonzero(active[feat]).flatten()
    support = support[torch.argsort(Y[feat, support].abs(), descending=True)].tolist()
    counts = n_active[support]

    sign = 1.0 if Y[feat, support].sum() >= 0 else -1.0
    proj = sign * proj_all[:, feat]
    proj_z = (proj - proj.mean()) / proj.std()

    print(f"=== feature {feat}: on {len(support)} tokens ===")
    print(f"proj on z_{feat} over all of V*: mean={proj.mean():.3f}  std={proj.std():.3f}")
    print(f"features per token: mean={counts.mean():.2f} (all of V*: {n_active.mean():.2f})  "
          f"min={counts.min()}  max={counts.max()}  only this feature: {int((counts == 1).sum())}")
    dist = pd.DataFrame({
        f"feature {feat} tokens": pd.Series(counts).value_counts(normalize=True),
        "all V*": pd.Series(n_active).value_counts(normalize=True),
    }).fillna(0.0).sort_index()
    dist.index.name = "# active features"
    display(dist.style.format("{:.1%}"))

    rows, co = [], Counter()
    for j in support:
        terms = Y[:, j].abs() * z_norms
        others = [i for i in torch.nonzero(active[:, j]).flatten().tolist() if i != feat]
        others.sort(key=lambda i: -terms[i].item())
        co.update(others)
        rows.append({
            "token": tokenizer.convert_ids_to_tokens(vstar_ids[j]),
            "label": LABEL_TAG[codes[j]],
            f"Y[{feat}]": Y[feat, j].item(),
            "n_active": int(n_active[j]),
            "share": (terms[feat] / terms.sum()).item(),
            "proj": proj[j].item(),
            "proj / avg feature": (proj[j] / avg_feat_proj[j]).item(),
            "proj / random dir": (proj[j] / rand_proj[j]).item(),
            "proj z vs V*": proj_z[j].item(),
            "other features (Y)": ", ".join(f"{i} ({Y[i, j].item():+.2f})" for i in others),
        })
    per_token = pd.DataFrame(rows)
    per_token.index += 1
    with pd.option_context("display.max_rows", None):
        display(per_token)

    # Which other features co-occur most often on this feature's tokens.
    co_df = pd.DataFrame([
        {"feature": i, "shared tokens": n, "frac of support": n / len(support),
         "rho": rho.get(i, float("nan")), "in_F_C": i in F_C}
        for i, n in co.most_common(10)
    ])
    print(f"most frequent co-active features on feature {feat}'s tokens:")
    display(co_df)

In [ ]:
# All tokens in V*, sorted by proj / avg feature (descending). Uses the variables computed in the cell above.
SHOW_ROWS = 100  # None shows all |V*| rows

terms_all = Y.abs() * z_norms[:, None]            # (k, |V*|): size of each term |Y_{j,t}| ||z_j||
token_strs = tokenizer.convert_ids_to_tokens(vstar_ids)

for feat in F_C:
    sign = 1.0 if Y[feat][active[feat]].sum() >= 0 else -1.0
    proj = sign * proj_all[:, feat]
    proj_z = (proj - proj.mean()) / proj.std()

    all_tokens = pd.DataFrame({
        "token": token_strs,
        "label": [LABEL_TAG[c] for c in codes],
        "in_support": active[feat].numpy(),
        # f"Y[{feat}]": Y[feat].numpy(),
        # "n_active": n_active,
        "share": (terms_all[feat] / terms_all.sum(0)).numpy(),
        # "proj": proj.numpy(),
        "proj / avg feature": (proj / avg_feat_proj).numpy(),
        # "proj / random dir": (proj / rand_proj).numpy(),
        "proj z vs V*": proj_z.numpy(),
    }).sort_values("proj / avg feature", ascending=False, ignore_index=True)
    all_tokens.index += 1

    n_sup = int(active[feat].sum())
    print(f"=== feature {feat}: all {len(all_tokens)} tokens of V*, sorted by proj / avg feature ===")
    # print(f"support tokens among the top {n_sup}: {int(all_tokens['in_support'].head(n_sup).sum())} / {n_sup}")
    with pd.option_context("display.max_rows", None):
        display(all_tokens if SHOW_ROWS is None else all_tokens.head(SHOW_ROWS))